# Machine Learning Project 1 — Chained Bike-Demand Pipeline

This notebook is the final presentation layer. Core implementation lives in `src/`.

## Phase 1 — Gradient Descent

### Expectation

_Write this **before** running the phase._

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config
from src.data import compute_team_seed, load_data, seeded_split
from src.features import Phase1Features
from src.metrics import regression_metrics
from src.phase1_gradient_descent import (
    LinearRegressionGD,
    largest_curvature,
    load_phase1_artifact,
    save_phase1_artifact,
)
from src.visualization import plot_gd_runs, plot_shift_by_hour

# Team seed: first reproduce the brief's example, then compute ours.
assert compute_team_seed(["34521", "40218", "41190"]) == 41698
seed = compute_team_seed(config.TEAM_IDS)
assert seed == config.EXPECTED_SEED
print("Team seed:", seed)

# The only seeded split: 80% train / 20% validation.
train_all, test_X = load_data(config.TRAIN_PATH, config.TEST_PATH)
train_df, val_df = seeded_split(train_all, seed, config.VALIDATION_SIZE)
y_train = train_df[config.TARGET].to_numpy(dtype=float)
y_val = val_df[config.TARGET].to_numpy(dtype=float)
print(f"train: {len(train_df)} rows, validation: {len(val_df)} rows")

### Phase 1 feature choices and justification

`Phase1Features` (in `src/features.py`) builds every column below. Its one-hot levels and its scaler are fitted on the 8,708 training rows only, then applied unchanged to validation (and, at the end, to `test.csv`). The evidence is in the two cells that follow.

| Column | Phase 1 representation | Why |
|---|---|---|
| `hr` | one-hot, 23 dummies (00:00 is the reference) | Demand has two weekday commute peaks and a weekend midday hump. A numeric `hr` would force one straight line across the day; one-hot gives every hour its own level. |
| `weekday` | one-hot, 6 dummies (Sunday is the reference) | Days of the week differ in level; the numbers 0–6 are labels, not amounts. |
| `holiday` | 0/1 | Holidays are rare but can change the daily pattern. |
| `workingday` | **not used** | In our training rows `workingday` = (Mon–Fri) − `holiday` with 0 exceptions (holidays only fall on Mon–Fri). With the weekday dummies, `holiday` and the intercept it is an exact linear combination, so the weights would not be unique. We keep `weekday` + `holiday` because they carry strictly more (Saturday vs Sunday, Monday vs Friday); `workingday` can be rebuilt from them for Phase 2. |
| `weathersit` | one-hot, levels 2 and 3 (1 = clear is the reference); level 4 is merged into 3 | Our training portion has 0 rows with level 4 (the only labeled one fell into validation), so no coefficient can be learned for it. Level 4 is the most severe level, closest to 3; left unmapped, it would silently be treated as clear weather. |
| `mnth` | one-hot, 11 dummies (January is the reference) | Seasonal representation. Every month in our training rows has exactly one `season`, so `season` is a coarser copy of `mnth` and the two cannot be used together. Month did better on validation (R² 0.6823 vs 0.6805) and separates months that season groups together, e.g. January from March. |
| `season` | **not used** | Exact function of `mnth` here (see above). |
| `yr` | 0/1 | Demand grows from 2011 to 2012. |
| `temp`, `hum`, `windspeed` | numeric | Weather measurements. Phase 1 is linear in them; non-linear effects are Phase 2's job. |
| `atemp` | **not used** | Correlation 0.984 with `temp` in our training rows, so it is a near-copy; its verdict is tested properly in Phase 4. |
| `dteday` | used through `yr`, `mnth`, `weekday`; no extra column | Its year, month and weekday agree exactly with `yr`, `mnth` and `weekday` (0 mismatches each), so those columns already carry them. Day-of-month is unusable: training has only days 1–19 and `test.csv` only day 20. What remains is elapsed time (`days_since_start`). It adds only 0.0005 R² (0.6829 vs 0.6823), but it is almost a combination of `yr` and month: the condition number jumps from 56 to 8,084 and GD needs 36,407 updates instead of 512. Not worth it in Phase 1. |
| `instant` | **not used** | Row identifier; kept only to align the submission file. |
| `cnt` | target | Never part of X. |

**Scaling.** Every feature column (dummies included) is standardized with its training mean and standard deviation; the intercept column stays at 1. All gradient components are then on comparable scales, so one learning rate suits every weight, and the learning rate can be stated relative to the curvature of the loss.

In [ ]:
# Facts behind the encoding choices, checked on the training portion.
dates = pd.to_datetime(train_df["dteday"])
mon_to_fri = train_df["weekday"].between(1, 5).astype(int)
encoding_facts = pd.Series({
    "rows where yr != year of dteday": (dates.dt.year - 2011 != train_df["yr"]).sum(),
    "rows where mnth != month of dteday": (dates.dt.month != train_df["mnth"]).sum(),
    "rows where weekday != weekday of dteday (0 = Sunday)":
        ((dates.dt.dayofweek + 1) % 7 != train_df["weekday"]).sum(),
    "rows where workingday != (Mon-Fri) - holiday":
        (mon_to_fri - train_df["holiday"] != train_df["workingday"]).sum(),
    "months that contain more than one season":
        (train_df.groupby("mnth")["season"].nunique() > 1).sum(),
    "rows with weathersit = 4": (train_df["weathersit"] == 4).sum(),
})
print(encoding_facts.to_string())
print(f"correlation of temp and atemp in train: {train_df['temp'].corr(train_df['atemp']):.3f}")
print("day of month in train:", dates.dt.day.min(), "to", dates.dt.day.max(),
      "| in test.csv:", sorted(pd.to_datetime(test_X["dteday"]).dt.day.unique()))

In [ ]:
# Compare the two seasonal representations, with and without the elapsed-time
# trend from dteday. Every design uses the same GD with step 1/largest curvature.
MAX_ITER = 50_000
TOL = 1e-4

design_rows = []
for seasonal in ("season", "mnth"):
    for use_trend in (False, True):
        candidate = Phase1Features(seasonal=seasonal, use_trend=use_trend).fit(train_df)
        X_tr, X_va = candidate.transform(train_df), candidate.transform(val_df)
        eigenvalues = np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr))
        model = LinearRegressionGD(learning_rate=1.0 / eigenvalues[-1],
                                   max_iter=MAX_ITER, tol=TOL)
        result = model.fit(X_tr, y_train)
        val_scores = regression_metrics(y_val, model.predict(X_va))
        design_rows.append({
            "seasonal": seasonal,
            "days_since_start": use_trend,
            "n_features": X_tr.shape[1] - 1,
            "condition_number": eigenvalues[-1] / eigenvalues[0],
            "status": result.status,
            "iterations": result.iterations,
            "train_r2": regression_metrics(y_train, model.predict(X_tr))["r2"],
            "val_r2": val_scores["r2"],
            "val_rmse": val_scores["rmse"],
        })
design_table = pd.DataFrame(design_rows)
design_table

### Learning rate, stopping rule and convergence evidence

**Loss and gradient.** $L(w) = \frac{1}{2n}\lVert Xw - y\rVert^2$, with gradient $\nabla L = \frac{1}{n}X^\top(Xw - y)$. `LinearRegressionGD.loss_and_gradient` computes it in this readable form. Because the loss is quadratic, the loop uses the equivalent precomputed form $\nabla L = Hw - b$ with $H = X^\top X/n$ and $b = X^\top y/n$, computed once per fit, so an update costs $p^2$ instead of $n \cdot p$ operations. Every fit asserts at its start and end that both forms give the same loss and gradient. Initialization: all weights zero. The run is deterministic, so re-running this notebook reproduces our weights exactly.

**Learning rate.** The loss is quadratic, so GD is stable exactly when the learning rate is below $2/\lambda_{max}$, where $\lambda_{max}$ is the largest eigenvalue of $X^\top X/n$ (the largest curvature of the loss). We therefore try multiples of $1/\lambda_{max}$ on both sides of that limit: 0.005, 0.1, 0.5, 1, 1.9 and 2.1. Rule: among the runs that converge, take the one that needs the fewest updates.

**Stopping rule** (the same for every run):
- *converged*: largest |gradient entry| < tol = 1e-4. At the minimum the gradient is exactly zero, so a small gradient is the proof of convergence. A flat loss curve is not: a run that crawls also looks flat.
- *diverged*: the loss becomes non-finite or exceeds 10⁶ × the starting loss.
- *cap*: 50,000 updates.

**Evidence** (outputs below):
- 2.1/λ_max **diverges**: stopped after 84 updates with the loss at 3.7·10¹⁰.
- 0.005/λ_max is **too slow**: it hits the 50,000-update cap with max |gradient| 0.0104, about 100× above tol, although its loss curve already looks flat.
- The four rates in between all **converge** to the same loss (4933.995). 1.9/λ_max (learning rate 0.9551) needs only 267 updates and its loss never increases, so it is our choice.
- tol = 1e-4 is tight enough: GD's validation predictions are within 0.0074 bikes of the exact least-squares minimum. That is an independent check, not used for training.

In [ ]:
# Chosen representation (justified above): month dummies, no elapsed-time trend.
SEASONAL, USE_TREND = "mnth", False
features = Phase1Features(seasonal=SEASONAL, use_trend=USE_TREND).fit(train_df)
X_train, X_val = features.transform(train_df), features.transform(val_df)

lam_max = largest_curvature(X_train)
print(f"{X_train.shape[1] - 1} features + intercept; largest curvature = {lam_max:.4f}; "
      f"GD is stable only for learning rate < 2/curvature = {2 / lam_max:.4f}")

# Learning rates as multiples of 1/curvature, on both sides of the 2/curvature limit.
MULTIPLIERS = [0.005, 0.1, 0.5, 1.0, 1.9, 2.1]
sweep = {}
for multiplier in MULTIPLIERS:
    model = LinearRegressionGD(learning_rate=multiplier / lam_max, max_iter=MAX_ITER, tol=TOL)
    sweep[f"{multiplier:g}/curvature"] = model.fit(X_train, y_train)

sweep_table = pd.DataFrame([{
    "step": label,
    "learning_rate": result.learning_rate,
    "status": result.status,
    "iterations": result.iterations,
    "loss_ever_increased": bool(np.any(np.diff(result.loss_history) > 0)),
    "final_loss": result.loss_history[-1],
    "final_max_abs_gradient": result.grad_history[-1],
} for label, result in sweep.items()])
display(sweep_table)
plot_gd_runs(sweep, TOL, "Phase 1 learning-rate sweep (MSE loss, training portion)")
plt.show()

In [ ]:
# Final Phase 1 model: the converged learning rate that needed the fewest updates.
converged = {label: r for label, r in sweep.items() if r.converged}
chosen_step = min(converged, key=lambda label: converged[label].iterations)
phase1_model = LinearRegressionGD(learning_rate=converged[chosen_step].learning_rate,
                                  max_iter=MAX_ITER, tol=TOL)
phase1_result = phase1_model.fit(X_train, y_train)
assert phase1_result.converged

phase1_metrics = {
    "train": regression_metrics(y_train, phase1_model.predict(X_train)),
    "validation": regression_metrics(y_val, phase1_model.predict(X_val)),
}
print(f"step {chosen_step}: learning rate {phase1_result.learning_rate:.5f}, "
      f"{phase1_result.iterations} updates, final max|gradient| "
      f"{phase1_result.grad_history[-1]:.2e}")
display(pd.DataFrame(phase1_metrics).T)

# Independent check (not used for training): the exact least-squares minimum.
# If GD had stalled, its predictions would differ noticeably from these.
exact_weights = np.linalg.lstsq(X_train, y_train, rcond=None)[0]
gap = np.max(np.abs(X_val @ phase1_result.weights - X_val @ exact_weights))
print(f"largest validation prediction gap between GD and exact minimum: {gap:.4f} bikes")

save_phase1_artifact(
    config.PHASE1_ARTIFACT, features, phase1_result,
    settings={"seed": seed, "max_iter": MAX_ITER, "tol": TOL,
              "step": chosen_step, "largest_curvature": lam_max},
    metrics=phase1_metrics,
)
print("Saved for Phase 2:", config.PHASE1_ARTIFACT.relative_to(config.ROOT))

### Operator check: errors at busy hours

R² treats every hour alike, but the operator's costly failure is an empty station at a busy time. So for every fitted model we also report, on validation:
- errors per decile of demand (decile edges from training demand);
- the **top-decile shortfall**, i.e. the mean bikes missing in hours whose actual demand is above the training 90th percentile;
- the hour × day-type cells where the shortfall concentrates.

Grouping by *actual* demand answers the operator's question directly, but it flatters no model: the top bin also collects hours that were busy partly by chance, so any imperfect model looks under-predicted there. Grouping by *predicted* demand has no such selection effect. Under-prediction that persists there is a genuine bias.

In [ ]:
from src.operator_checks import (
    demand_decile_edges,
    errors_by_demand_decile,
    errors_by_hour_daytype,
    top_decile_shortfall,
    unscaled_coefficients,
)
from src.visualization import plot_shortfall_by_hour

# Operator check: are the busiest hours under-supplied? Decile edges come from
# training demand; error = predicted - actual, shortfall = bikes missing.
decile_edges = demand_decile_edges(y_train)
pred_phase1_val = phase1_model.predict(X_val)
print("training demand decile edges (bikes):", decile_edges[1:-1])
print(f"top-decile shortfall (actual > {decile_edges[-2]:.0f} bikes): "
      f"{top_decile_shortfall(y_val, pred_phase1_val, decile_edges):.1f} bikes per hour")
print("validation hours predicted below zero:", int(np.sum(pred_phase1_val < 0)))

print("\nBy ACTUAL demand decile:")
display(errors_by_demand_decile(y_val, pred_phase1_val, decile_edges).round(2))
print("By PREDICTED demand decile (no selection effect):")
display(errors_by_demand_decile(y_val, pred_phase1_val, decile_edges, bin_on="predicted").round(2))

phase1_hour_errors = errors_by_hour_daytype(val_df, y_val, pred_phase1_val)
print("Hour x day-type cells with the largest shortfall:")
display(phase1_hour_errors.sort_values("mean_shortfall_bikes", ascending=False).head(10).round(1))
plot_shortfall_by_hour({"Phase 1 (MSE)": phase1_hour_errors},
                       "Phase 1: mean bikes short per hour (validation)")
plt.show()

### Outcome

_Fill this from the actual pipeline results._

## Phase 1 bonus — asymmetric cost (optional, +5%)

This section is separate from the chain. It reads the Phase 1 artifact but writes nothing that Phase 2 or the submission file reads (Check 3 below proves the artifact is unchanged). It uses the same split (seed 36954), the same 47 features and fitted scaler, and the same `LinearRegressionGD` loop. Only the loss/gradient hook is replaced (`AsymmetricLinearRegressionGD` in `src/bonus_asymmetric.py`). This loss is not quadratic (each row's weight depends on the sign of its residual), so it has no fixed $H = X^\top X/n$: the bonus evaluates the explicit gradient $X^\top(c \odot r)/n$ at every update instead of Phase 1's precomputed form.

**Our reading of "under-predicting costs 3× as much".** We keep the squared error and weight under-predictions by $k = 3$:

$$\ell(r) = \tfrac{1}{2}\,c(r)\,r^2,\qquad r = \hat y - y,\qquad c(r) = \begin{cases} k & r < 0 \text{ (under-prediction)}\\ 1 & r \ge 0 \end{cases}$$

Why this reading:
- The brief asks us to minimize the loss *instead of MSE*, derive *its gradient*, and compare against *our MSE model*. Keeping the quadratic shape changes exactly one thing, the asymmetry, so the shift we measure is caused by the asymmetry alone.
- The loss is differentiable everywhere, so our Phase 1 GD and its gradient-based stopping rule work unchanged.

The other reading, a cost linear in the number of bikes (pinball loss, as in the newsvendor problem, whose optimum is the 75th percentile), is arguably closer to how lost rentals add up. But it has no gradient at $r = 0$, and its slope never shrinks toward zero, so fixed-step GD with our stopping rule could not converge on it. Our assumption is stated openly: a large shortfall is penalized more than in proportion to its size.

**Gradient derivation.**

$$L_k(w) = \frac{1}{2n}\sum_{i=1}^{n} c_i\, r_i^2,\qquad r_i = x_i^\top w - y_i$$

1. *Derivative of one row's loss.* For $r < 0$, $\ell(r) = \tfrac{k}{2}r^2$, so $\ell'(r) = k\,r$. For $r > 0$, $\ell'(r) = r$. At $r = 0$ both one-sided difference quotients $(\ell(h) - \ell(0))/h = \tfrac{1}{2}c\,h$ tend to $0$, so $\ell'(0) = 0$. Hence $\ell'(r) = c(r)\,r$ everywhere, and it is continuous.
2. *Chain rule.* With $\partial r_i/\partial w = x_i$:
$$\nabla L_k(w) = \frac{1}{n}\sum_{i=1}^{n} c_i\, r_i\, x_i = \frac{1}{n}X^\top (c \odot r)$$
3. *Comparison with MSE.* Since $c(r)\,r = r + (k-1)\min(r, 0)$:
$$\nabla L_k = \underbrace{\tfrac{1}{n}X^\top r}_{\text{MSE gradient}} + \tfrac{k-1}{n}X^\top \min(r, 0)$$
This is the MSE gradient plus an extra push that comes only from under-predicted rows. With $k = 1$ it is exactly Phase 1.
4. *Convexity.* $\ell'$ is continuous and non-decreasing (slope $k$ on the left of 0, slope 1 on the right), so $\ell$ is convex. $L_k$ is an average of convex functions of $w$, so any point with zero gradient is the global minimum.
5. *Step size.* $|\ell'(a) - \ell'(b)| \le k\,|a - b|$, so the gradient of $L_k$ changes at most $k$ times as fast as in the MSE case, and GD is guaranteed to converge for learning rates below $2/(k\,\lambda_{max})$. The safe limit is $k$ times smaller than for MSE.
6. *What the minimum means.* The intercept row of $X^\top(c \odot r) = 0$ gives $\sum_{r_i>0} r_i = k\sum_{r_i<0}|r_i|$: on training, total over-prediction equals 3× total under-prediction. For an intercept-only model the minimizer is the 0.75 *expectile* ($\tau = k/(k+1)$), an asymmetric mean, not the 75th percentile.

### Expectation (bonus)

_Write this **before** running the bonus cells._

In [ ]:
from src.bonus_asymmetric import (
    AsymmetricLinearRegressionGD,
    finite_difference_gradient,
    prediction_summary,
    residual_structure,
    shift_by_hour,
)

UNDER_COST = 3.0  # k: under-predicting costs 3x as much as over-predicting

# Read Phase 1's artifact (read-only) and fingerprint it so we can prove later
# that the bonus did not change what Phase 2 consumes.
artifact_hash_before = hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest()
phase1 = load_phase1_artifact(config.PHASE1_ARTIFACT)
bonus_features = Phase1Features.from_dict(phase1["features"])
Xb_train, Xb_val = bonus_features.transform(train_df), bonus_features.transform(val_df)
w_phase1 = phase1["weights"]
assert np.array_equal(Xb_train, X_train), "bonus must use exactly the Phase 1 features"

# Check 1: analytic gradient X^T (c * r) / n against central finite differences.
checker = AsymmetricLinearRegressionGD(under_cost=UNDER_COST)
for name, w in (("at Phase 1 weights", w_phase1), ("at zero weights", np.zeros_like(w_phase1))):
    analytic = checker.loss_and_gradient(Xb_train, y_train, w)[1]
    numeric = finite_difference_gradient(checker, Xb_train, y_train, w)
    error = np.max(np.abs(analytic - numeric)) / np.max(np.abs(analytic))
    print(f"Check 1 {name}: largest relative gradient error = {error:.1e}")

# Check 2: with k = 1 the bonus model is the Phase 1 model, so from the same
# start and settings it must reach the Phase 1 weights. Phase 1's loop uses the
# precomputed H w - b and the bonus the explicit gradient, so the two agree up
# to floating-point rounding rather than bit for bit.
k_one = AsymmetricLinearRegressionGD(
    under_cost=1.0, learning_rate=phase1["learning_rate"],
    max_iter=phase1["settings"]["max_iter"], tol=phase1["settings"]["tol"],
).fit(Xb_train, y_train)
weight_gap = np.max(np.abs(k_one.weights - w_phase1))
print(f"Check 2: k = 1 reaches the Phase 1 weights: largest difference {weight_gap:.1e}; "
      f"{k_one.iterations} updates vs Phase 1's {phase1['iterations']}")
assert weight_gap < 1e-8

**Learning rate, start and stopping rule.**
- *Stopping rule and tol:* identical to Phase 1.
- *Start:* the Phase 1 weights. The feature space is the same, and by convexity (step 4) the start does not change the answer.
- *Learning rate:* step 5 says the safe limit shrinks by $k$. We run the Phase 1 learning rate unchanged, and the Phase 1 learning rate divided by $k$. The second is 0.955/3 = 0.318, below the guaranteed limit $2/(3\lambda_{max}) = 0.335$.

In [ ]:
# Learning rate: the Phase 1 step, and the Phase 1 step divided by k (the
# gradient can change up to k times faster, so the safe step shrinks by k).
# Both start from the Phase 1 weights.
phase1_step = phase1["learning_rate"]
bonus_runs = {}
for label, step in (("Phase 1 step", phase1_step), ("Phase 1 step / k", phase1_step / UNDER_COST)):
    model = AsymmetricLinearRegressionGD(under_cost=UNDER_COST, learning_rate=step,
                                         max_iter=MAX_ITER, tol=TOL)
    bonus_runs[label] = model.fit(Xb_train, y_train, init_weights=w_phase1)

display(pd.DataFrame([{
    "step": label,
    "learning_rate": result.learning_rate,
    "status": result.status,
    "iterations": result.iterations,
    "loss_ever_increased": bool(np.any(np.diff(result.loss_history) > 0)),
    "final_loss": result.loss_history[-1],
    "final_max_abs_gradient": result.grad_history[-1],
} for label, result in bonus_runs.items()]))
plot_gd_runs(bonus_runs, TOL, "Bonus: asymmetric loss (k = 3), warm start from Phase 1 weights")
plt.show()

asym_result = bonus_runs["Phase 1 step / k"]
assert asym_result.converged

In [ ]:
# How the fitted predictions shift (validation set, same rows for both models).
pred_mse = Xb_val @ w_phase1
pred_asym = Xb_val @ asym_result.weights

comparison = pd.DataFrame({
    "MSE model (Phase 1)": prediction_summary(y_val, pred_mse, UNDER_COST),
    "Asymmetric model (k=3)": prediction_summary(y_val, pred_asym, UNDER_COST),
})
comparison["change"] = comparison.iloc[:, 1] - comparison.iloc[:, 0]
display(comparison)

shift = pred_asym - pred_mse
print(f"shift per hour (bikes): mean {shift.mean():.1f}, median {np.median(shift):.1f}, "
      f"min {shift.min():.1f}, max {shift.max():.1f}; "
      f"share of hours shifted down: {np.mean(shift < 0):.1%}")

hour_table = shift_by_hour(val_df, y_val, pred_mse, pred_asym)
plot_shift_by_hour(hour_table, "Mean predictions by hour: MSE vs asymmetric (validation)")
plt.show()

config.BONUS_DIR.mkdir(parents=True, exist_ok=True)
comparison.to_csv(config.BONUS_DIR / "validation_comparison.csv")
hour_table.to_csv(config.BONUS_DIR / "shift_by_hour.csv", index=False)

In [ ]:
# Underfitting vs demand variability. Phase 1 has no hour x day-type terms, so
# its residuals in each hour x day-type cell need not average to zero.
train_residual = Xb_train @ w_phase1 - y_train
cell_table, shares = residual_structure(train_df, train_residual)
print(f"Phase 1 training residual variance: {shares['between_cell_share']:.1%} between "
      f"hour x day-type cells (systematic misfit), {shares['within_cell_share']:.1%} within cells")

# Where does the shift land? Cells with a positive mean residual are ones the
# MSE model already over-predicts; raising them further only adds cost.
val_cells = (val_df[["workingday", "hr"]].assign(shift=shift)
             .merge(cell_table, on=["workingday", "hr"], how="left"))
already_over = val_cells["mean_residual"] > 0
print(f"validation hours in already over-predicted cells: {already_over.mean():.1%}, "
      f"receiving {val_cells['shift'][already_over].sum() / val_cells['shift'].sum():.1%} "
      f"of the total upward shift")
print(f"mean shift there: {val_cells['shift'][already_over].mean():.1f} bikes, "
      f"elsewhere: {val_cells['shift'][~already_over].mean():.1f} bikes")

cells = hour_table.merge(cell_table, on=["workingday", "hr"])
print(f"correlation across the 48 cells: shift vs within-cell residual SD = "
      f"{cells['shift'].corr(cells['within_sd']):.2f}, "
      f"shift vs cell mean residual = {cells['shift'].corr(cells['mean_residual']):.2f}")
cells.sort_values("shift", ascending=False).head(8).round(1)

In [ ]:
# Operator check for the bonus: does k = 3 fix the busiest hours, or lift everything?
for label, pred in (("MSE model", pred_mse), ("Asymmetric model", pred_asym)):
    print(f"{label}: top-decile shortfall "
          f"{top_decile_shortfall(y_val, pred, decile_edges):.1f} bikes per hour")

by_decile_mse = errors_by_demand_decile(y_val, pred_mse, decile_edges)
by_decile_asym = errors_by_demand_decile(y_val, pred_asym, decile_edges)
display(pd.DataFrame({
    "decile": by_decile_mse["decile"],
    "mean_actual": by_decile_mse["mean_actual"],
    "mean_shift": by_decile_asym["mean_predicted"] - by_decile_mse["mean_predicted"],
    "under_rate_mse": by_decile_mse["under_prediction_rate"],
    "under_rate_asym": by_decile_asym["under_prediction_rate"],
    "shortfall_mse": by_decile_mse["mean_shortfall_bikes"],
    "shortfall_asym": by_decile_asym["mean_shortfall_bikes"],
}).round(2))

# Which coefficients moved? In original units: a dummy's coefficient is the
# bikes it adds relative to its reference level (hr=0, Sunday, January, clear).
names, mean, std = bonus_features.feature_names, bonus_features.mean, bonus_features.std
coefficients = pd.DataFrame({
    "mse": unscaled_coefficients(w_phase1, names, mean, std),
    "asym": unscaled_coefficients(asym_result.weights, names, mean, std),
})
coefficients["change"] = coefficients["asym"] - coefficients["mse"]
largest_moves = coefficients["change"].abs().sort_values(ascending=False).index[:12]
display(coefficients.loc[largest_moves].round(1))

plot_shortfall_by_hour({
    "MSE model": errors_by_hour_daytype(val_df, y_val, pred_mse),
    "Asymmetric model (k=3)": errors_by_hour_daytype(val_df, y_val, pred_asym),
}, "Mean bikes short per hour: MSE vs asymmetric (validation)")
plt.show()

In [ ]:
# Check 3: the bonus left Phase 1's artifact (what Phase 2 consumes) untouched.
assert hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest() == artifact_hash_before
assert np.array_equal(load_phase1_artifact(config.PHASE1_ARTIFACT)["weights"],
                      phase1_result.weights)
print("Check 3: Phase 1 artifact and weights unchanged by the bonus")

### Outcome (bonus)

_Draft written from the run above. Rewrite it in your own words before submission._

**Optimization.**
- The Phase 1 learning rate (0.955) **diverged** after 12 updates under $k = 3$. The Phase 1 rate divided by $k$ (0.318) converged in 388 updates, and its loss never increased.
- All three code checks pass:
  - the analytic gradient matches finite differences (relative error ≤ 2.7·10⁻¹⁰);
  - $k = 1$ reaches the Phase 1 weights (equal up to floating-point rounding, see Check 2);
  - the Phase 1 artifact is unchanged.

**How the predictions shift** (validation, 2,178 hours):

| | MSE model (Phase 1) | Asymmetric model ($k=3$) |
|---|---:|---:|
| R² | 0.6823 | 0.6059 |
| RMSE | 103.56 | 115.35 |
| asymmetric cost ($k=3$) | 11,145 | 9,343 (−16.2%) |
| mean prediction | 196.4 | 239.0 |
| hours under-predicted | 45.8% | 32.2% |
| mean shortfall (bikes/hour) | 37.2 | 21.2 |
| mean surplus (bikes/hour) | 39.6 | 66.1 |

- Predictions rise by 42.6 bikes/hour on average (median 39.2, range −2.7 to +108.7). Only 0.2% of hours go down.
- For the operator: 16.0 fewer bikes short per hour, at the price of 26.6 more idle bikes per hour. When a missing bike costs 3× an idle one this is a good trade, and the 16% lower asymmetric cost measures it.
- R² and RMSE get worse, as expected, because they score the MSE objective.
- 32.2% of hours are still under-predicted, not 25%. That is consistent with step 6: the loss targets an asymmetric mean, not the 75th percentile.

**How much of the shift is Phase 1 underfitting, and how much is genuine demand variability?**
- **Size of the underfitting.** Phase 1 has only main effects. 44.4% of its training residual variance lies *between* hour × day-type cell means. That is systematic error a model with hour × working-day terms would remove. The remaining 55.6% (*within* cells) is the most that can be genuine variability, and it still contains other missing effects (e.g. hour × temperature). So underfitting accounts for at least 44% of the residual spread the loss reacts to.
- **Underfitting misplaces the shift.** The model cannot raise weekday 08:00 without raising weekend 08:00 too.
  - 50.5% of validation hours fall in cells the MSE model already over-predicts on average. They still receive 48.6% of the total upward shift (+41.0 bikes/hour, vs +44.2 elsewhere).
  - Worst case: weekend 08:00, already over-predicted (383 vs an actual mean of 106), is raised by another 98 to 482.
- **Some of the gain is correcting underfitting, not hedging uncertainty.** MSE under-predicts the working-day peaks (08:00: 374 vs 480; 17:00: 465 vs 533). The asymmetric model lands close to the actual means (456 and 536), mostly because it shifts everything up, not because it learned the peaks.
- **The variability-driven part.** Across the 48 cells, the shift correlates 0.67 with the within-cell residual spread and only 0.14 with the cell's mean residual. The loss responds to spread, not to where the bias is: variable commute hours get the largest buffers (+70 to +98 bikes), working-day late-evening and night hours (21:00–05:00) the smallest (+17 to +29).

**Busiest hours (operator check).**
- The top-decile shortfall (hours above 452 actual bikes) falls from 176.6 to 112.1 bikes per hour (−37%). But 88% of those hours are still under-predicted, versus 96% before.
- The lift grows with demand, but only mildly: +29.5 bikes in the quietest decile, +69.5 in the busiest. Much of it is a broad lift. The raw intercept rises by 36.9 bikes, which applies to every hour, so even 03:00 gets more bikes.
- The targeted part sits in the hour dummies: 08:00 +57.2, 18:00 +44.2, 17:00 +43.5, and weekend-midday hours (12:00–15:00) +18 to +25. The Mon and Fri dummies fall by 17 and 22 bikes relative to Sunday, so the model leans toward weekends. This is the loss chasing Phase 1's largest misses, at weekend 11:00–15:00 (120–154 bikes short) and working-day 08:00/17:00 (107–118 short). With main effects only, it can reach them only by moving whole hours and whole days.

**Conclusion.** With Phase 1's main-effects features, about half of the extra bikes go to hours that were already over-stocked, and that part is an artifact of underfitting. A clean estimate of the buffer that genuine demand variability justifies needs a model without this underfitting, i.e. the same bonus run on Phase 2's features. The asymmetric loss shifts errors toward surplus, but it cannot create structure the features lack.

## Phase 2 — Polynomial Regression

### Which interactions? Residual diagnosis of Phase 1 (training rows only)

Phase 1 has main effects only, so each training residual $e = y - \hat y$ contains whatever the main effects cannot express. For example, if the 08:00 level depends on whether it is a working day, Phase 1's 08:00 residuals will be positive on working days and negative on the others.

For each candidate combination we group the 8,708 training residuals into cells and measure:
- **between share**: the share of Phase 1's residual variance that the cell means explain, i.e. how much of what Phase 1 misses this combination could recover. Overlapping combinations (e.g. month and temperature) are not additive.
- **adjusted share**: the between share corrected for the number of cells, like adjusted R², so a combination with many cells cannot win by size alone.
- **cell support**: rows per cell, and how many cells have fewer than 30 rows (a 30-row mean is still uncertain by about ±18 bikes).

Continuous columns are cut into equal-count bins on training rows: 10 bins for `temp` alone, 5 bins inside products. Validation rows are not used. Phase 1's residuals average to zero within every hour (checked below), so an hour × X cell measures only the interaction.

In [ ]:
from src.phase2_polynomial import residual_cell_means, residual_diagnosis
from src.visualization import plot_residual_profiles

# Phase 2 starts from Phase 1's artifact: its features, scaler and weights.
phase1_artifact = load_phase1_artifact(config.PHASE1_ARTIFACT)
phase1_features = Phase1Features.from_dict(phase1_artifact["features"])
phase1_residual = y_train - phase1_features.transform(train_df) @ phase1_artifact["weights"]
hour_means = pd.Series(phase1_residual).groupby(train_df["hr"].to_numpy()).mean()
print(f"mean training residual {phase1_residual.mean():.4f}; "
      f"largest |mean residual| of any single hour {hour_means.abs().max():.4f}")
print("holiday dates in the training portion:",
      train_df.loc[train_df["holiday"] == 1, "dteday"].nunique())

diagnosis = residual_diagnosis(train_df, phase1_residual)
config.PHASE2_DIR.mkdir(parents=True, exist_ok=True)
diagnosis.to_csv(config.PHASE2_DIR / "residual_diagnosis.csv", index=False)
display(diagnosis.round(4))

panels = [
    ("hour x workingday", residual_cell_means(train_df, phase1_residual, ["hr", "workingday"]),
     "workingday", False),
    ("hour x yr", residual_cell_means(train_df, phase1_residual, ["hr", "yr"]), "yr", False),
    ("temperature quintile x hour",
     residual_cell_means(train_df, phase1_residual, ["temp_bin5", "hr"]), "temp_bin5", True),
    ("hour x season", residual_cell_means(train_df, phase1_residual, ["hr", "season"]),
     "season", False),
]
plot_residual_profiles(panels, "Phase 1 training residuals by hour (positive = under-predicted)")
plt.show()

### Re-ranking after the confirmed groups

The Step A shares overlap (month, season and temperature all follow the yearly cycle), so we re-rank the remaining candidates on what is left *after* the confirmed groups:
1. remove the hour × workingday group means from Phase 1's training residuals;
2. remove hour × workingday and hour × yr together (least squares on both groups' indicators; for one group this is just its group means).

For a continuous candidate (hour × temp, temp × hum, temp²) the share is the R² of the remaining residuals regressed on its columns; for a group candidate this R² is exactly the between-group share of Step A. Training rows only.

In [ ]:
from src.phase2_polynomial import (
    RERANK_CANDIDATES,
    diagnosis_frame,
    project_out,
    remove_group_means,
    rerank,
)

frame = diagnosis_frame(train_df)
phase1_total = np.sum(phase1_residual ** 2)
stages = {
    "after hour x workingday":
        remove_group_means(phase1_residual, frame, [["hr", "workingday"]]),
    "after hour x workingday + hour x yr":
        remove_group_means(phase1_residual, frame, [["hr", "workingday"], ["hr", "yr"]]),
}
for stage, residual in stages.items():
    remaining = np.sum((residual - residual.mean()) ** 2) / phase1_total
    print(f"{stage}: {remaining:.1%} of Phase 1's residual variance remains")
    table = rerank(residual, frame)
    table.insert(3, "share_of_phase1_variance", table["share"] * remaining)
    display(table.round(4))

# Season and temperature both follow the yearly cycle: what does each add after the other?
after_confirmed = stages["after hour x workingday + hour x yr"]
overlap_rows = []
for first, second in [("hour x season", "hour x temp"),
                      ("hour x temp", "hour x season"),
                      ("hour x season", "hour x workingday x temp"),
                      ("hour x season", "hour x workingday x season")]:
    residual = project_out(after_confirmed, frame, RERANK_CANDIDATES[first])
    remaining = np.sum((residual - residual.mean()) ** 2) / phase1_total
    added = rerank(residual, frame, {second: RERANK_CANDIDATES[second]}).iloc[0]
    overlap_rows.append({
        "already in": first,
        "candidate": second,
        "adds_share_of_phase1_variance": added["share"] * remaining,
        "adjusted_share_of_remaining": added["adjusted_share"],
    })
display(pd.DataFrame(overlap_rows).round(4))

### Phase 2 design: which terms, how they are formed, how they are fitted

**Groups, in the order they are added** (evidence: the two tables above).

| Degree | Group | Columns | Why |
|---|---|---:|---|
| 2 | hour × workingday | 23 | Largest Step A share (0.444); the weekday/weekend daily shape. |
| 2 | hour × yr | 23 | 0.107 in Step A; 2012 peaks are higher, not just shifted. |
| 2 | hour × season | 69 | Best yearly-cycle candidate after the confirmed groups (adjusted 0.2055 vs hour × temp 0.1822). After it, hour × temp adds only 0.0061 of Phase 1's variance, while season still adds 0.0283 after temp. No sparse cells (min 77 rows). Month was rejected in Step A (105 sparse cells). |
| 2 | hour-block × weathersit | 8 | Day parts: night 0–5, morning 6–9, midday 10–15, evening 16–19, late 20–23. Adjusted 0.0373 with 15 cells and none sparse, vs full hour × weathersit 0.0420 with 72 cells, 13 of them sparse. |
| 2 | temp × hum | 1 | Adjusted 0.0323 for a single extra column (temp and hum are already in Phase 1). |
| 3 | hour × workingday × yr | 24 | Still 0.0525 adjusted after both confirmed groups: the weekday peaks grow more from 2011 to 2012 than the rest of the day. Includes workingday × yr. Min 117 rows per term. |
| 3 | hour × workingday × temp | 47 | Adds 0.0246 after hour × season with no sparse cells, more than hour × workingday × season (0.0229, 59 sparse cells). Includes hour × temp and workingday × temp, so the model stays hierarchical. |

Rejected after re-ranking: hour × temp as a separate degree-2 group (overlaps season; it enters only inside the degree-3 group), full hour × weathersit (sparse), temp² (adjusted 0.0029).

**How terms are formed.**
- Explicit term lists only (`group_terms` in `src/phase2_polynomial.py`). No product of two levels of one column (always 0), no square of a 0/1 column, and reference levels are skipped, so no term duplicates a Phase 1 column.
- `workingday` re-enters only through its interactions; its main effect is already carried by `weekday` + `holiday`.
- A product is built from raw values. The hour (or hour-block) indicator is used as 0/1; every other factor is first centered at its training mean (e.g. `hr=8 * (workingday − its training mean)`). Centering does not change which models the columns can express, but it makes the products much less correlated with the hour indicators: the full design's condition number with and without centering is shown below.
- Phase 1's 47 columns keep Phase 1's fitted scaler unchanged. Each new column is standardized with its own training mean and standard deviation.

**How it is fitted.**
- Degree = interaction order. Degree 1 is Phase 1 exactly; degree 2 adds the five pair groups; degree 3 adds the two triple groups. Groups are added one at a time and every step is scored, so each group's contribution is a number.
- Every step starts from Phase 1's weights in their own columns and 0 for every new column (the brief's warm start). Before training we assert that this starting model reproduces Phase 1's predictions.
- Same GD code, loss, stopping rule (max |gradient| < 1e-4, cap 50,000) and learning-rate rule as Phase 1: the same multiple of 1/λ_max, with λ_max recomputed for each expanded matrix.
- Degree is chosen on validation R²; the top-decile shortfall (by actual and by predicted demand) is reported next to it for the operator.

In [ ]:
from src.phase2_polynomial import PHASE2_GROUPS, Phase2Features

# Design checks on training rows only (nothing is fitted here).
phase1_train_pred = phase1_features.transform(train_df) @ phase1_artifact["weights"]
design_rows, groups_so_far = [], []
for group in [None, *PHASE2_GROUPS]:
    if group:
        groups_so_far.append(group)
    candidate = Phase2Features(phase1_features, groups_so_far).fit(train_df)
    X_tr = candidate.transform(train_df)
    eigenvalues = np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr))
    start = X_tr @ candidate.warm_start(phase1_artifact["weights"])
    design_rows.append({
        "added group": group or "(Phase 1)",
        "degree": PHASE2_GROUPS.get(group, 1),
        "columns": X_tr.shape[1],
        "rank": np.linalg.matrix_rank(X_tr),
        "condition_number": eigenvalues[-1] / eigenvalues[0],
        "warm_start_max_prediction_change": np.max(np.abs(start - phase1_train_pred)),
    })
display(pd.DataFrame(design_rows))

uncentered = Phase2Features(phase1_features, list(PHASE2_GROUPS), center_factors=False).fit(train_df)
X_unc = uncentered.transform(train_df)
eig_unc = np.linalg.eigvalsh(X_unc.T @ X_unc / len(X_unc))
print(f"full design condition number: {design_rows[-1]['condition_number']:.0f} with centering, "
      f"{eig_unc[-1] / eig_unc[0]:.0f} without")

print("Training rows supporting each new term (non-zero before centering):")
display(candidate.term_support(train_df).groupby("group", sort=False)["support_rows"]
        .agg(terms="count", min_rows="min", median_rows="median"))

### Expectation

_Written before the official run._

1. **Accuracy.** Phase 1 reached validation R² 0.682. Its residual diagnosis showed 44% of its error is systematic between hour × working-day groups, and a further 10% between hour × year groups. We therefore expect a large gain: roughly 0.85–0.90 at degree 2 and 0.90–0.93 at degree 3. For comparison, a similar ~200-column design reached 0.89 on a harder chronological holdout during exploration.

2. **Largest single gain.** Hour × workingday, since it is Phase 1's biggest blind spot (the 08:00 weekend over-prediction of ~250 bikes). We expect it alone to lift validation R² by roughly 0.10–0.15.

3. **Overfitting.** 243 columns against ~8,700 training rows, with every term supported by at least 79 rows. We expect the train/validation gap to grow only slightly from Phase 1's 0.015, staying below about 0.03. Larger growth would point to a specific sparse group.

4. **Peak shortfall.** Phase 1's busiest-predicted hours came in 96 bikes short, because an additive model cannot reach the peaks. Hour × workingday and hour × year should raise commute and 2012 peaks, so we expect this shortfall to fall by at least half. It will not reach zero, because demand scales multiplicatively and interactions only approximate that.

5. **Convergence.** The same 1.9/λmax rule should converge. With centred interactions (condition number 1,713) we expect more updates than Phase 1's 267, but well within the cap.

In [ ]:
from src.phase2_polynomial import (
    PHASE2_GROUPS,
    Phase2Features,
    fit_and_score,
    save_phase2_artifact,
)

# Same rule as Phase 1: the same multiple of 1/largest curvature, same stopping rule.
settings1 = phase1_artifact["settings"]
STEP_MULTIPLIER = phase1_artifact["learning_rate"] * settings1["largest_curvature"]
print(f"learning rate = {STEP_MULTIPLIER:.2f} / largest curvature; "
      f"tol {settings1['tol']}, cap {settings1['max_iter']} updates")

# Add the groups one at a time; every step starts from the Phase 1 weights.
sequence_rows, fitted_steps, groups_so_far = [], {}, []
for group in [None, *PHASE2_GROUPS]:
    if group:
        groups_so_far.append(group)
    step_features = Phase2Features(phase1_features, groups_so_far).fit(train_df)
    result, row = fit_and_score(step_features, train_df, y_train, val_df, y_val,
                                phase1_artifact["weights"], STEP_MULTIPLIER,
                                settings1["max_iter"], settings1["tol"], decile_edges)
    label = group or "(Phase 1)"
    fitted_steps[label] = (step_features, result)
    sequence_rows.append({"added group": label, "degree": PHASE2_GROUPS.get(group, 1), **row})
sequence = pd.DataFrame(sequence_rows)
config.PHASE2_DIR.mkdir(parents=True, exist_ok=True)
sequence.to_csv(config.PHASE2_DIR / "phase2_sequence.csv", index=False)

print("Scores after each added group (validation unless marked train):")
display(sequence[["added group", "degree", "n_features", "train_r2", "val_r2",
                  "train_val_r2_gap", "val_rmse", "top_decile_shortfall",
                  "top_predicted_decile_shortfall", "top_predicted_decile_mean_error",
                  "negative_predictions"]].round(4))
print("Convergence of each fit:")
display(sequence[["added group", "rank_ok", "condition_number", "learning_rate", "status",
                  "iterations", "final_max_abs_gradient", "gap_to_exact_bikes"]])

In [ ]:
# A degree's model is the last step of that degree (all of its groups added).
degree_rows = {degree: sequence[sequence["degree"] == degree].iloc[-1] for degree in (1, 2, 3)}
chosen_degree = max(degree_rows, key=lambda degree: degree_rows[degree]["val_r2"])
chosen_row = degree_rows[chosen_degree]
phase2_features, phase2_result = fitted_steps[chosen_row["added group"]]
display(pd.DataFrame(degree_rows).T[["added group", "n_features", "train_r2", "val_r2",
                                     "train_val_r2_gap", "val_rmse", "top_decile_shortfall",
                                     "top_predicted_decile_shortfall", "status", "iterations"]])
print(f"Chosen degree (highest validation R2): {chosen_degree}")

# Operator checks for the three degree models.
degree_predictions = {}
for degree, row in degree_rows.items():
    step_features, step_result = fitted_steps[row["added group"]]
    degree_predictions[f"degree {degree}"] = step_features.transform(val_df) @ step_result.weights

def by_decile(bin_on, column):
    """One column per degree model; a decile no hour falls into stays empty."""
    columns = {}
    for label, pred in degree_predictions.items():
        table = errors_by_demand_decile(y_val, pred, decile_edges, bin_on=bin_on)
        columns[label] = pd.Series(table[column].to_numpy(), index=table["decile"].astype(int))
    return pd.DataFrame(columns).rename_axis("decile")

print("Mean shortfall (bikes) by ACTUAL demand decile:")
display(by_decile("actual", "mean_shortfall_bikes").round(1))
print("Under-prediction rate by ACTUAL demand decile:")
display(by_decile("actual", "under_prediction_rate").round(2))
print("Mean error (predicted - actual) by PREDICTED demand decile (no selection effect):")
display(by_decile("predicted", "mean_error").round(1))
print("Mean shortfall (bikes) by PREDICTED demand decile:")
display(by_decile("predicted", "mean_shortfall_bikes").round(1))

hour_errors = {label: errors_by_hour_daytype(val_df, y_val, pred)
               for label, pred in degree_predictions.items()}
print(f"Hour x day-type cells with the largest shortfall, degree {chosen_degree}:")
display(hour_errors[f"degree {chosen_degree}"]
        .sort_values("mean_shortfall_bikes", ascending=False).head(10).round(1))
plot_shortfall_by_hour(hour_errors, "Mean bikes short per hour by degree (validation)")
plt.show()

save_phase2_artifact(
    config.PHASE2_ARTIFACT, phase2_features, phase2_result, chosen_degree,
    chosen_row.to_dict(), sequence,
    settings={"seed": seed, "step_multiplier": STEP_MULTIPLIER,
              "max_iter": settings1["max_iter"], "tol": settings1["tol"]},
)
print("Saved for Phase 3:", config.PHASE2_ARTIFACT.relative_to(config.ROOT))

In [ ]:
# Validation R2 is slightly ABOVE training R2 at degree 2 and 3. Is that a concern?
# Compare errors directly (RMSE), and see how much R2 moves between random
# validation-sized subsets of training rows under the same fitted model.
chosen_X_train = phase2_features.transform(train_df)
chosen_train_pred = chosen_X_train @ phase2_result.weights
chosen_val_pred = phase2_features.transform(val_df) @ phase2_result.weights
train_rmse = regression_metrics(y_train, chosen_train_pred)["rmse"]
val_rmse = regression_metrics(y_val, chosen_val_pred)["rmse"]
print(f"degree {chosen_degree}: RMSE train {train_rmse:.2f}, validation {val_rmse:.2f}; "
      f"target variance validation/train = {y_val.var() / y_train.var():.3f}")

rng = np.random.default_rng(seed)
train_residual = y_train - chosen_train_pred
subset_r2 = []
for _ in range(1000):
    rows = rng.choice(len(y_train), size=len(y_val), replace=False)
    subset_r2.append(1 - np.sum(train_residual[rows] ** 2)
                     / np.sum((y_train[rows] - y_train[rows].mean()) ** 2))
low, high = np.percentile(subset_r2, [5, 95])
print(f"R2 of 1,000 random {len(y_val)}-row subsets of training rows: "
      f"5th-95th percentile {low:.4f} to {high:.4f}")

### Degree choice and justification

**Chosen: degree 3** (242 features: Phase 1's 47, five pair groups, two triple groups).

| | Degree 1 | Degree 2 | Degree 3 |
|---|---:|---:|---:|
| validation R² | 0.6823 | 0.9009 | **0.9225** |
| validation RMSE | 103.56 | 57.85 | **51.14** |
| top-decile shortfall, by actual demand (bikes) | 176.6 | 73.6 | **58.9** |
| top-decile shortfall, by predicted demand (bikes) | 134.9 | 53.8 | **43.3** |
| mean error in the top predicted decile (bikes) | −95.9 | −23.8 | **−11.8** |

- Degree 3 has the highest validation R², the rule set before the run. It is also the best model for the operator on every peak measure, so the two criteria agree.
- Every group added helped on validation. The smallest step is temp × hum: +0.0005 R² and −0.1 bikes of top-decile shortfall for one column, effectively nothing. We keep it because Phase 4's L1 penalty is the principled place to drop it.
- No sign of overfitting at this complexity. Training and validation RMSE are almost equal (cell above). Validation R² is slightly higher only because the validation target varies a little more, and the gap is well inside how much R² moves between random validation-sized subsets.
- Every fit converged under the same rule as Phase 1. Degree 3 needed 3,884 updates, and its predictions are within 0.06 bikes of the exact least-squares solution.

Phase 3 tests this formally: it varies complexity around degree 3 and re-evaluates on a chronological split.

### Outcome

_Fill this from the actual pipeline results._

### Bonus follow-up: the asymmetric loss on Phase 2's features

The main bonus (Phase 1 section) stays on Phase 1's features, as the brief asks. This short follow-up reruns the same $k = 3$ model on the features Phase 2 chose, to separate the part of the Phase 1 shift caused by underfitting from the buffer that genuine demand variability justifies.
- *Same code and checks:* `AsymmetricLinearRegressionGD`, a finite-difference gradient check at the Phase 2 weights, and $k = 1$ must reach the Phase 2 weights (up to floating-point rounding) from Phase 2's own start and settings.
- *Same learning-rate rule as the main bonus:* the model's MSE learning rate divided by $k$, warm start from its MSE weights, same stopping rule.
- *Read-only:* the Phase 1 and Phase 2 artifacts must be unchanged afterwards.

### Expectation (bonus follow-up)

_Written before the official run._

With hour × workingday captured, the asymmetric model should no longer lift hours that Phase 1 already over-predicted, such as weekend 08:00. We expect a smaller average upward shift than the 42.6 bikes/h on Phase 1 features, concentrated on hours whose demand varies genuinely (working-day commute peaks, weekend afternoons), and a lower top-decile shortfall than before.

In [ ]:
from src.bonus_asymmetric import buffer_by_hour
from src.operator_checks import top_predicted_decile_shortfall
from src.phase2_polynomial import Phase2Features, load_phase2_artifact

phase1_hash = hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest()
phase2_hash = hashlib.sha256(config.PHASE2_ARTIFACT.read_bytes()).hexdigest()
phase2_artifact = load_phase2_artifact(config.PHASE2_ARTIFACT)
followup_features = Phase2Features.from_dict(phase2_artifact["features"])
Xf_train, Xf_val = followup_features.transform(train_df), followup_features.transform(val_df)
w_phase2 = phase2_artifact["weights"]
settings2 = phase2_artifact["settings"]
print(f"Phase 2 features: degree {phase2_artifact['degree']}, {Xf_train.shape[1] - 1} columns")

# Check 1: analytic gradient against central finite differences at the Phase 2 weights.
analytic = checker.loss_and_gradient(Xf_train, y_train, w_phase2)[1]
numeric = finite_difference_gradient(checker, Xf_train, y_train, w_phase2)
print(f"Check 1: largest relative gradient error = "
      f"{np.max(np.abs(analytic - numeric)) / np.max(np.abs(analytic)):.1e}")

# Check 2: k = 1 from Phase 2's own start and settings reaches the Phase 2
# weights (up to floating-point rounding, as in the main bonus).
k_one_phase2 = AsymmetricLinearRegressionGD(
    under_cost=1.0, learning_rate=phase2_artifact["learning_rate"],
    max_iter=settings2["max_iter"], tol=settings2["tol"],
).fit(Xf_train, y_train, init_weights=followup_features.warm_start(phase1_artifact["weights"]))
weight_gap = np.max(np.abs(k_one_phase2.weights - w_phase2))
print(f"Check 2: k = 1 reaches the Phase 2 weights: largest difference {weight_gap:.1e}; "
      f"{k_one_phase2.iterations} updates vs Phase 2's {phase2_artifact['iterations']}")
assert weight_gap < 1e-8

# The k = 3 fit: Phase 2's learning rate divided by k, warm start from the Phase 2 weights.
followup_result = AsymmetricLinearRegressionGD(
    under_cost=UNDER_COST, learning_rate=phase2_artifact["learning_rate"] / UNDER_COST,
    max_iter=settings2["max_iter"], tol=settings2["tol"],
).fit(Xf_train, y_train, init_weights=w_phase2)
print(f"k = 3 fit: {followup_result.status} after {followup_result.iterations} updates, "
      f"final max|gradient| {followup_result.grad_history[-1]:.2e}, loss ever increased: "
      f"{bool(np.any(np.diff(followup_result.loss_history) > 0))}")
assert followup_result.converged

In [ ]:
# Phase 1 features vs Phase 2 features: where the k = 3 buffer goes (validation).
pred_mse2 = Xf_val @ w_phase2
pred_asym2 = Xf_val @ followup_result.weights

buffers = buffer_by_hour(val_df, y_val, pred_mse, pred_asym).merge(
    buffer_by_hour(val_df, y_val, pred_mse2, pred_asym2),
    on=["workingday", "hr"], suffixes=("_phase1_features", "_phase2_features"))
buffers = buffers[["workingday", "hr",
                   "shift_phase1_features", "shift_phase2_features",
                   "short_phase1_features", "short_phase2_features",
                   "idle_phase1_features", "idle_phase2_features"]]
print("Mean upward shift and the asymmetric model's bikes short / idle per hour (validation):")
display(buffers.round(1))
buffers.to_csv(config.BONUS_DIR / "followup_buffer_by_hour.csv", index=False)

followup_models = {
    "Phase 1 features, MSE": pred_mse,
    "Phase 1 features, k=3": pred_asym,
    "Phase 2 features, MSE": pred_mse2,
    "Phase 2 features, k=3": pred_asym2,
}
display(pd.DataFrame({
    label: {
        **prediction_summary(y_val, pred, UNDER_COST),
        "top_decile_shortfall": top_decile_shortfall(y_val, pred, decile_edges),
        "top_predicted_decile_shortfall":
            top_predicted_decile_shortfall(y_val, pred, decile_edges),
    }
    for label, pred in followup_models.items()
}).round(3))

# Check 3: both artifacts are unchanged by the follow-up.
assert hashlib.sha256(config.PHASE1_ARTIFACT.read_bytes()).hexdigest() == phase1_hash
assert hashlib.sha256(config.PHASE2_ARTIFACT.read_bytes()).hexdigest() == phase2_hash
print("Check 3: Phase 1 and Phase 2 artifacts unchanged by the follow-up")

### Outcome (bonus follow-up)

_Fill this from the actual pipeline results._

## Phase 3 — Bias–Variance Tradeoff

### Phase 3 plan: what we vary, what we measure, how we decide

**Consumes** Phase 2's artifact: its groups, in its order, and its chosen design.

**1. Complexity sweep** (seeded split). Phase 1 → each Phase 2 group in Phase 2's order → then *beyond* Phase 2, to reach the high-variance side:
- hour × holiday, hour × weekday, hour × month and full hour × weathersit (the groups rejected in Phase 2 Step A);
- a degree-4 group: hour × workingday × yr × season, with the lower-order terms it needs.

Some extension columns are exact combinations of earlier columns: hour × month contains hour × season, full hour × weathersit contains the hour blocks, and hour × weekday plus hour × holiday reproduce hour × workingday. Those columns are detected on the training rows (QR factorization, columns in order) and dropped. The count is reported.

Each fit uses the same GD, learning-rate rule (1.9/λ_max), stopping rule and cap as Phases 1 and 2:
- Phase 2's steps start from the Phase 1 weights exactly as in Phase 2, so they must reproduce Phase 2's recorded scores (asserted).
- The extensions start from Phase 2's chosen fit plus zeros.
- A fit that hits the cap is **flagged as not converged and never called over-fitted**. Its exact least-squares score is shown for reference only.

**2. What tells bias from variance.**
- *Sweep:* on the bias side, train and validation improve together and the gap stays near 0. On the variance side, train keeps improving while validation stalls or falls, and the gap opens.
- *Learning curves* (train/validation RMSE vs training rows; 25%, 50%, 75%, 100% of the training portion; validation fixed), for Phase 2's design and the largest extended design. Curves that converge with a small, flat gap mean bias-limited: more data would not help. A wide gap that closes as data grows means variance-limited.

**3. The chronological split** (the only other split). Hold out the last 20% of labeled *days*, so whole days stay on one side; the cut uses dates only. Everything is refitted on the earlier days: encoder levels, scaler, centering means, the Phase 1 GD, and every sweep step.
- The seeded validation hours share days, weather and periods with training hours: it measures interpolation.
- The chronological holdout lies after all training data, in a later and busier stretch of 2012: it measures extrapolation in time.
- The hidden test (day 20 of every month, both years) is neither: its days are unseen but sit inside the training period.

**4. Operator checks** for every fit: top-decile shortfall by actual and by predicted demand.

**5. Decision rules, fixed before the run.**
- *Target for Phase 4:* among converged fits on the seeded split, the one with the fewest features whose validation R² is within `R2_TOLERANCE` of the best.
- *Noise candidates:* groups in the target design whose addition raised validation R² by less than `R2_TOLERANCE` on **both** splits **and** lowered the seeded top-decile shortfall by less than `SHORTFALL_TOLERANCE` bikes.

In [ ]:
from src.data import chronological_split
from src.phase2_polynomial import load_phase2_artifact
from src.visualization import plot_complexity_sweep, plot_learning_curves, plot_split_comparison
from src.phase3_bias_variance import (
    SWEEP_EXTENSIONS,
    choose_target,
    fit_phase1,
    group_support,
    learning_curve,
    run_sweep,
    save_phase3_artifact,
    step_contributions,
    sweep_steps,
)

# Phase 3 consumes Phase 2's artifact: its groups, order and chosen design.
phase2_artifact = load_phase2_artifact(config.PHASE2_ARTIFACT)
steps = sweep_steps(phase2_artifact)
display(pd.DataFrame([{"step": i, "added group": s["added group"], "degree": s["degree"],
                       "part": s["part"], "groups so far": len(s["groups"]),
                       "starts from": s.get("start_from", "Phase 1 weights")}
                      for i, s in enumerate(steps)]))

# Extension designs on the seeded training rows only (nothing fitted here).
extension_rows = []
for step in steps:
    if step["part"] != "extension":
        continue
    design = Phase2Features(phase1_features, step["groups"]).fit(train_df)
    X_tr = design.transform(train_df)
    eigenvalues = np.linalg.eigvalsh(X_tr.T @ X_tr / len(X_tr))
    extension_rows.append({"added group": step["added group"], "degree": step["degree"],
                           "n_features": X_tr.shape[1] - 1,
                           "dropped_so_far": len(design.dropped),
                           "condition_number": eigenvalues[-1] / eigenvalues[0],
                           **group_support(design, train_df, step["added group"])})
display(pd.DataFrame(extension_rows))

# The chronological split: dates only.
chrono_train, chrono_val, first_validation_day = chronological_split(train_all)
print(f"chronological split: {chrono_train['dteday'].nunique()} earlier days "
      f"({len(chrono_train)} rows) | {chrono_val['dteday'].nunique()} later days "
      f"({len(chrono_val)} rows), from {first_validation_day} to {chrono_val['dteday'].max()}")

### Expectation

_Written before the official run._

1. **Where seeded validation peaks.** We expect a plateau, not a sharp peak: the extensions should move seeded validation R² by only a few thousandths around Phase 2's 0.9225. On a random split, hours from the same day fall in both training and validation, so even sparse terms (hour × holiday, hour × month) partly "recognise" validation days, which hides overfitting. If any extension visibly hurts, we expect it to be hour × holiday first (23 terms on 7–30 rows each) or degree 4 (216 extra columns).

2. **Train/validation gap.** Training R² keeps rising with every extension, because a larger linear model can never fit its own training data worse, while validation stays flat. The gap therefore widens from about −0.005 to roughly +0.01–0.02 at 818 features.

3. **Convergence.** The condition numbers rise about 3× (1,713 → 5,530). GD updates grow roughly in proportion, from 3,884 to around 10,000–15,000, so we expect every fit to converge within the 50,000 cap.

4. **Learning curves.** The Phase 2 design should be bias/noise-limited: training and validation RMSE close together and flattening near 51, so more data would barely help. The 818-feature design should be variance-limited: at 25% of the data (~2,200 rows for 818 weights) a wide gap, shrinking as data is added.

5. **Chronological vs seeded.** The chronological holdout (Aug–Dec 2012) is the busiest period and lies after everything the model trained on. We expect validation R² around 0.89–0.91 for the degree-3 design, with RMSE around 65–75 bikes against 51 on the seeded split; exploration on an almost identical boundary gave 0.89 / 71 bikes for a similar ~200-feature design. The extensions should hurt more here than on the seeded split, especially degree 4: the earlier part contains no 2012 autumn hours, so the year × season terms must extrapolate to a combination never seen in training.
   Which estimate to trust: the chronological score for forecasting genuinely future periods; the seeded score is optimistic because it shares days with training. The hidden test (day 20 of every month, both years) lies between the two: unseen days, but inside the period the final model is fitted on. We therefore expect test performance slightly below the seeded estimate and above the chronological one.

6. **Operator checks.** Larger peak shortfalls on the chronological split than on the seeded one, since a model fitted before the late-2012 growth under-predicts its busiest hours.

7. **Diagnosis and target.** At degree 3 the model is bias/noise-limited, not variance-limited, so the target stays the Phase 2 degree-3 design (243 features). Noise candidates: temp × hum (+0.0005 R² in Phase 2) among current terms; the extensions (hour × holiday, hour × month, degree 4) are where variance enters.

**Decision rules, fixed before the run:** the target is the converged fit with the fewest features whose seeded validation R² is within 0.002 of the best. A group is a noise candidate only if it gains < 0.002 R² on both splits AND reduces the peak shortfall by < 1 bike.

In [ ]:
# The sweep on the seeded split, with the same rules as Phase 2.
settings2 = phase2_artifact["settings"]
STEP3, MAX_ITER3, TOL3 = settings2["step_multiplier"], settings2["max_iter"], settings2["tol"]
seeded_sweep, seeded_fits = run_sweep(steps, train_df, y_train, val_df, y_val,
                                      phase1_features, phase1_artifact["weights"],
                                      STEP3, MAX_ITER3, TOL3)

# Integrity: Phase 2's steps must reproduce the scores Phase 2 recorded.
recorded = pd.DataFrame(phase2_artifact["sequence"]).set_index("added group")["val_r2"]
reproduced = seeded_sweep.set_index("added group").loc[recorded.index, "val_r2"]
assert np.allclose(recorded, reproduced, rtol=0, atol=1e-10)
print("Phase 2 steps reproduce Phase 2's recorded validation R2")

config.PHASE3_DIR.mkdir(parents=True, exist_ok=True)
seeded_sweep.to_csv(config.PHASE3_DIR / "sweep_seeded.csv", index=False)
print("Scores and operator checks (validation unless marked train):")
display(seeded_sweep[["added group", "part", "degree", "n_features", "train_r2", "val_r2",
                      "train_val_r2_gap", "train_rmse", "val_rmse", "top_decile_shortfall",
                      "top_predicted_decile_shortfall"]].round(4))
print("Convergence (exact_val_r2 = exact least-squares fit, reference only):")
display(seeded_sweep[["added group", "n_features", "dropped_columns", "condition_number",
                      "status", "iterations", "final_max_abs_gradient", "gap_to_exact_bikes",
                      "val_r2", "exact_val_r2"]])

converged = seeded_sweep[seeded_sweep["status"] == "converged"]
peak = converged.loc[converged["val_r2"].idxmax()]
print(f"validation peak (converged fits): {peak['added group']}, "
      f"{peak['n_features']} features, R2 {peak['val_r2']:.4f}")
first_extension = seeded_sweep[seeded_sweep["part"] == "extension"]["n_features"].min()
last_phase2 = seeded_sweep[seeded_sweep["part"] == "Phase 2"]["n_features"].max()
plot_complexity_sweep(seeded_sweep, "Complexity sweep on the seeded split",
                      peak["n_features"], (first_extension + last_phase2) / 2)
plt.show()

In [ ]:
# Learning curves: refit everything on growing subsets of the training portion.
FRACTIONS = [0.25, 0.5, 0.75, 1.0]
phase1_state = phase1_artifact["features"]
curve_designs = {
    "Phase 2's chosen design": list(phase2_artifact["groups"]),
    "largest extended design": steps[-1]["groups"],
}
curves = {}
for label, groups in curve_designs.items():
    curves[label] = learning_curve(groups, phase1_state["seasonal"], phase1_state["use_trend"],
                                   train_df, y_train, val_df, y_val, FRACTIONS, seed,
                                   STEP3, MAX_ITER3, TOL3)
    print(label)
    display(curves[label].round(3))
pd.concat(curves, names=["design"]).to_csv(config.PHASE3_DIR / "learning_curves.csv")
plot_learning_curves(curves, "Learning curves (validation set fixed)")
plt.show()

In [ ]:
# The same sweep on the chronological split, everything refitted on the earlier days.
yc_train = chrono_train[config.TARGET].to_numpy(dtype=float)
yc_val = chrono_val[config.TARGET].to_numpy(dtype=float)
chrono_phase1, chrono_phase1_result = fit_phase1(
    chrono_train, yc_train, phase1_state["seasonal"], phase1_state["use_trend"],
    STEP3, MAX_ITER3, TOL3)
assert chrono_phase1_result.converged
chrono_sweep, chrono_fits = run_sweep(steps, chrono_train, yc_train, chrono_val, yc_val,
                                      chrono_phase1, chrono_phase1_result.weights,
                                      STEP3, MAX_ITER3, TOL3)
chrono_sweep.to_csv(config.PHASE3_DIR / "sweep_chronological.csv", index=False)

comparison = pd.DataFrame({
    "added group": seeded_sweep["added group"],
    "n_features (seeded)": seeded_sweep["n_features"],
    "seeded_val_r2": seeded_sweep["val_r2"],
    "chrono_train_r2": chrono_sweep["train_r2"],
    "chrono_val_r2": chrono_sweep["val_r2"],
    "seeded_val_rmse": seeded_sweep["val_rmse"],
    "chrono_val_rmse": chrono_sweep["val_rmse"],
    "chrono_status": chrono_sweep["status"],
    "seeded_top_decile_shortfall": seeded_sweep["top_decile_shortfall"],
    "chrono_top_decile_shortfall": chrono_sweep["top_decile_shortfall"],
    "chrono_top_predicted_decile_shortfall": chrono_sweep["top_predicted_decile_shortfall"],
    "chrono_top_predicted_decile_mean_error": chrono_sweep["top_predicted_decile_mean_error"],
})
display(comparison.round(4))
plot_split_comparison(seeded_sweep, chrono_sweep, "Seeded vs chronological validation")
plt.show()

In [ ]:
# Decision rules fixed before the run (see the plan above and the Expectation).
R2_TOLERANCE = 0.002        # validation R2 differences smaller than this count as ties
SHORTFALL_TOLERANCE = 1.0   # bikes of top-decile shortfall

target = choose_target(seeded_sweep, R2_TOLERANCE)
target_features, target_result = seeded_fits[target["added group"]]
print(f"Target for Phase 4: design up to '{target['added group']}' "
      f"({target['n_features']} features, validation R2 {target['val_r2']:.4f})")

contributions = step_contributions(seeded_sweep, chrono_sweep)
display(contributions.round(4))
in_target = contributions["added group"].isin(target_features.groups)
noise = contributions[in_target
                      & (contributions["seeded_val_r2_change"] < R2_TOLERANCE)
                      & (contributions["chrono_val_r2_change"] < R2_TOLERANCE)
                      & (contributions["seeded_top_decile_shortfall_change"] > -SHORTFALL_TOLERANCE)]
noise_candidates = noise["added group"].tolist()
print("Noise candidates for the Phase 4 penalty:", noise_candidates or "none")

largest = seeded_sweep.iloc[-1]
diagnosis = {
    "seeded_validation_peak": {"step": peak["added group"], "n_features": int(peak["n_features"]),
                               "val_r2": float(peak["val_r2"])},
    "target_train_val_r2_gap": float(target["train_val_r2_gap"]),
    "largest_train_val_r2_gap": float(largest["train_val_r2_gap"]),
    "largest_status": largest["status"],
    "learning_curve_final_gap_rmse": {
        label: float(table["val_rmse"].iloc[-1] - table["train_rmse"].iloc[-1])
        for label, table in curves.items()},
    "target_val_r2": {"seeded": float(target["val_r2"]),
                      "chronological": float(chrono_sweep.set_index("added group")
                                             .loc[target["added group"], "val_r2"])},
    "r2_tolerance": R2_TOLERANCE,
    "shortfall_tolerance": SHORTFALL_TOLERANCE,
}
save_phase3_artifact(
    config.PHASE3_ARTIFACT, target, target_features, target_result.weights, diagnosis,
    noise_candidates,
    tables={"seeded_sweep": seeded_sweep, "chronological_sweep": chrono_sweep,
            "contributions": contributions},
    settings={"seed": seed, "first_chronological_validation_day": first_validation_day,
              "step_multiplier": STEP3, "max_iter": MAX_ITER3, "tol": TOL3},
)
print("Saved for Phase 4:", config.PHASE3_ARTIFACT.relative_to(config.ROOT))

### Outcome

_Fill this from the actual pipeline results._

## Phase 4 — Regularization

### Phase 4 plan: objective, search, selection and verdicts

**Consumes** Phase 3's artifact: the 564-feature target design (encoder, scaler, centering means, feature names, lineage), its diagnosis (bias-limited at degree 3; variance enters only past the target), and the noise candidate (temp × hum). Same seeded split and scaler.

**One objective for all three methods** (intercept $w_0$ never penalized):
$$J(w) = \frac{1}{2n}\lVert Xw - y\rVert^2 + \lambda\Big(\rho\sum_{j\ge1}|w_j| + (1-\rho)\sum_{j\ge1}\tfrac{w_j^2}{2}\Big)$$
Ridge: $\rho = 0$. Lasso: $\rho = 1$. Elastic Net: $\rho \in \{0.1, 0.5, 0.9\}$.

**Algorithm** (`PenalizedGD`, the Phase 1 loop with $H = X^\top X/n$, $b = X^\top y/n$ computed once per fit):
1. Gradient step on the smooth part: $H w - b + \lambda(1-\rho)\,w_{\text{pen}}$. Ridge's penalty simply adds $\lambda(1-\rho)$ to the diagonal of $H$, except the intercept's.
2. Soft-thresholding for the L1 part: each penalized weight is pulled toward 0 by $\eta\lambda\rho$ and set to **exactly** 0 if it would cross 0.
- *Learning rate:* $L = \lambda_{max}(H) + \lambda(1-\rho)$ bounds the curvature of the smooth part.
  - Ridge is plain gradient descent on a quadratic, so it keeps Phases 1–3's rule, $1.9/L$ (stable for any step below $2/L$).
  - Lasso and Elastic Net use $1/L$: soft-thresholding steps of at most $1/L$ are guaranteed never to increase the objective. Every fit reports `objective_ever_increased` as the check.
- *Stopping:* the largest entry of $(w - w_{next})/\eta$ falls below 1e-4, cap 50,000 updates. Without L1 this is exactly the gradient; with L1 it is 0 only at the minimum.
- *Checks:* tests show the code matches Ridge's closed-form solution and satisfies the Lasso optimality conditions.

**λ search: one path per method, from strong to weak.**
- Lasso and Elastic Net start at $\lambda_{max}$, the smallest λ at which every penalized weight is exactly 0, and go down to $10^{-4}\lambda_{max}$.
- Ridge has no exact-zero point, so its path runs from $10^{3}$ (every weight shrunk close to 0) down to $10^{-5}$ (essentially unpenalized).
- 30 log-spaced values per path. Each fit starts from the previous fit's weights, which makes the path cheap.
- **Revision after the first official run.** All four L1-type paths reached their best validation R² at the last grid point. For Lasso and Elastic Net 0.9 the curve had already flattened at the unpenalized level. For Elastic Net 0.5 (0.0007 short) and especially Elastic Net 0.1 (0.012 short, ending at λ = 0.07) the grid stopped before the peak, so the comparison between ratios was cut off. Every L1-type path now continues at the same spacing (4/29 of a decade) down to λ = 1e-5, Ridge's floor. Their first 30 values are unchanged. Ridge's grid is unchanged: it peaked inside.
- *Why this range:* Phase 3 found the target bias-limited, with variance only in its sparse terms. So we expect the best λ to be small, and the grid reaches far enough down to see the unpenalized end.

**Selection on the seeded validation set.** The brief fixes a single validation set for all tuning, so there is no k-fold cross-validation. A single holdout makes the chosen λ itself somewhat noisy, which is why near-ties count as ties. Only converged fits are eligible.
- Ridge: the highest validation R².
- Lasso and each Elastic Net ratio: the **largest** λ whose validation R² is within 0.002 of that path's best. Among equally good fits, this prefers the sparser, more stable one.
- Elastic Net overall: the ratio whose selected fit has the highest validation R² (ties: fewer nonzero weights).

**Fair comparison.** Same design, split, scaler, GD rule, stopping rule and grid density. Every fit also reports convergence and the operator checks.

**Column verdicts (all 14 original columns).** The *verdict model* is the selected fit with the highest validation R², refitted with its own λ and ρ on each modified design.
- *In the target* (hr, yr, mnth, season, holiday, weekday, workingday, weathersit, temp, hum, windspeed): grouped ablation. Rebuild the design without the column and every term derived from it, refit, and record ΔR², ΔRMSE and the peak-shortfall change. Overlapping terms that the column had made redundant come back in, so the ablation is fair.
- *Not in the target* (instant, dteday, atemp, plus the main effects of workingday and season): add the column back, refit, and record the change. Columns found to be exact combinations of the design are reported as such.
- *On its own:* the R² of a model with only that column, in its own full encoding, i.e. does it carry demand information at all? Categorical columns enter as their level indicators (hr as 23 hour dummies, never as one number 0–23), so a poor representation cannot hide a real signal. 0/1 and continuous columns enter as their values; dteday as elapsed days plus day of month.
- *Lasso evidence:* how many of the column's terms survive in the selected Lasso.
- *Rule (fixed before the run):*
  - **useful** if removing it costs at least 0.002 validation R² (or adding it back gains at least 0.002);
  - otherwise **redundant** if on its own it reaches R² ≥ 0.01 (its information is carried by other columns);
  - otherwise **uninformative**.

In [ ]:
from src.phase3_bias_variance import load_phase3_artifact
from src.phase4_regularization import (
    ADD_BACK_GROUPS,
    ORIGINAL_COLUMNS,
    alone_design,
    lambda_max_l1,
    refit_like,
    regularization_path,
    save_phase4_artifact,
    score_weights,
    select,
    verdict,
)
from src.visualization import plot_regularization_paths

# Phase 4 consumes Phase 3's artifact: the target design and its starting point.
phase3_artifact = load_phase3_artifact(config.PHASE3_ARTIFACT)
target_design = Phase2Features.from_dict(phase3_artifact["features"])
assert target_design.feature_names == phase3_artifact["feature_names"]
X4_train, X4_val = target_design.transform(train_df), target_design.transform(val_df)
print(f"target: up to '{phase3_artifact['target_step']}', {X4_train.shape[1] - 1} features; "
      f"Phase 3 noise candidates: {phase3_artifact['noise_candidates']}")

# Search settings (fixed before the run).
N_LAMBDAS = 30
L1_RATIOS = {"Lasso": 1.0, "Elastic Net 0.9": 0.9, "Elastic Net 0.5": 0.5, "Elastic Net 0.1": 0.1}
SELECT_R2_TOLERANCE = 0.002
VERDICT_R2_TOLERANCE = 0.002
ALONE_R2_THRESHOLD = 0.01
STEP4, MAX_ITER4, TOL4 = STEP3, MAX_ITER3, TOL3  # same GD rule as Phases 1-3

LAMBDA_FLOOR = 1e-5            # Ridge's smallest lambda; every path reaches it
L1_STEP_DECADES = 4 / 29       # spacing of the original 30-point, 4-decade L1 grids

grids = {"Ridge": (0.0, np.logspace(3, -5, N_LAMBDAS))}
for label, rho in L1_RATIOS.items():
    top = np.log10(lambda_max_l1(X4_train, y_train, rho))
    steps = int(np.ceil((top - np.log10(LAMBDA_FLOOR)) / L1_STEP_DECADES))
    grids[label] = (rho, 10 ** (top - L1_STEP_DECADES * np.arange(steps + 1)))
display(pd.DataFrame([{"method": label, "rho": rho, "largest lambda": grid[0],
                       "smallest lambda": grid[-1], "values": len(grid)}
                      for label, (rho, grid) in grids.items()]))

### Expectation

_Written before the official run._

1. **Accuracy.** Phase 3 showed train and validation error almost equal at 564 features (gap +0.0054), so little variance remains for a penalty to remove. We expect regularization to improve validation R² only barely, by a few thousandths at most above the unpenalized 0.9262, with the best λ values small.

2. **Sparsity.** Lasso should empty the weakest-supported groups first: temp × hum (it added only +0.0005 R² in Phase 2), hour × holiday (7–10 training rows per cell, and it slightly hurt on the seeded split), and the thin hour × month terms. Because so little variance remains, we expect most of the model to survive: roughly 400 or more of the 564 terms under our selection rule.

3. **Busiest hours.** Shrinking coefficients pulls predictions toward the average, which lowers the peaks. We therefore expect the busy-hour shortfall to get slightly worse under regularization, even where R² holds steady — a cost to the operator that R² alone would hide.

4. **Column verdicts** (selected predictions):
   - atemp: redundant (98.5% correlated with temp, so temp already carries its information)
   - workingday: useful (marginal correlation only 0.012, but hour × workingday was Phase 2's largest gain; removing it and everything built from it should cost real accuracy)
   - windspeed: uninformative (windspeed × hour scored below noise in the Phase 2 diagnosis)
   - instant: redundant (it rises with time, so yr and month already carry its information)

In [ ]:
# One warm-started path per method, from the largest lambda down.
config.PHASE4_DIR.mkdir(parents=True, exist_ok=True)
paths, path_weights = {}, {}
for label, (rho, grid) in grids.items():
    paths[label], path_weights[label] = regularization_path(
        X4_train, y_train, X4_val, y_val, grid, rho, STEP4, MAX_ITER4, TOL4, decile_edges)
    paths[label].to_csv(config.PHASE4_DIR / f"path_{label.replace(' ', '_').lower()}.csv",
                        index=False)
    print(label)
    display(paths[label][["lambda", "nonzero", "train_r2", "val_r2", "val_rmse",
                          "top_decile_shortfall", "top_predicted_decile_shortfall",
                          "status", "iterations", "objective_ever_increased"]].round(4))

In [ ]:
# Selection (rules above); only converged fits are eligible.
chosen = {"Ridge": select(paths["Ridge"], None)}
for label in L1_RATIOS:
    chosen[label] = select(paths[label], SELECT_R2_TOLERANCE)
elastic = [label for label in L1_RATIOS if label.startswith("Elastic Net")]
best_elastic = max(elastic, key=lambda label: (chosen[label]["val_r2"], -chosen[label]["nonzero"]))
final = {"Ridge": chosen["Ridge"], "Lasso": chosen["Lasso"], "Elastic Net": chosen[best_elastic]}

summary = pd.DataFrame(final).T[["lambda", "l1_ratio", "nonzero", "train_r2", "val_r2",
                                 "train_rmse", "val_rmse", "top_decile_shortfall",
                                 "top_predicted_decile_shortfall", "status", "iterations",
                                 "final_max_move"]]
summary.insert(3, "of_features", X4_train.shape[1] - 1)
# Each path must peak inside its grid, not at its last (weakest) lambda.
for label, path in paths.items():
    best = int(path["val_r2"].idxmax())
    print(f"{label}: best validation R2 {path['val_r2'].iloc[best]:.5f} at point {best + 1} "
          f"of {len(path)} (lambda {path['lambda'].iloc[best]:.3g})")
print(f"Elastic Net ratio chosen: {best_elastic}")
print("Every Elastic Net ratio at its selected lambda:")
display(pd.DataFrame({label: chosen[label] for label in elastic}).T[
    ["lambda", "nonzero", "val_r2", "val_rmse", "top_decile_shortfall"]])
print("Selected model per method (validation unless marked train):")
display(summary)
summary.to_csv(config.PHASE4_DIR / "selected_models.csv")

def weights_of(label, row):
    """Weights of the selected fit on its path."""
    position = int(np.flatnonzero(paths[label]["lambda"].to_numpy() == row["lambda"])[0])
    return path_weights[label][position]

final_weights = {"Ridge": weights_of("Ridge", chosen["Ridge"]),
                 "Lasso": weights_of("Lasso", chosen["Lasso"]),
                 "Elastic Net": weights_of(best_elastic, chosen[best_elastic])}
plot_regularization_paths(paths, {label: row["lambda"] for label, row in chosen.items()},
                          "Regularization paths (seeded validation)")
plt.show()

In [ ]:
# Column verdicts with the best selected model's settings.
verdict_method = max(final, key=lambda label: final[label]["val_r2"])
v_lambda, v_rho = final[verdict_method]["lambda"], final[verdict_method]["l1_ratio"]
v_weights = final_weights[verdict_method]
v_scores = score_weights(v_weights, X4_train, y_train, X4_val, y_val, decile_edges)
print(f"verdict model: {verdict_method} (lambda {v_lambda:.4g}, rho {v_rho}); "
      f"validation R2 {v_scores['val_r2']:.4f}")

lasso_weights = final_weights["Lasso"]
names = target_design.feature_names
lineage = target_design.lineage
target_groups = list(phase3_artifact["target_groups"])
rows = []
for column in ORIGINAL_COLUMNS:
    terms = [i for i, name in enumerate(names) if column in lineage[name]]
    in_model = len(terms) > 0
    row = {"column": column, "terms_in_target": len(terms),
           "terms_surviving_lasso": int(np.count_nonzero(lasso_weights[terms])) if terms else 0}
    ablation_change = add_back_change = np.nan
    all_dropped = False
    if in_model:  # remove the column and every term derived from it
        ablated = Phase2Features(target_design.phase1, target_groups,
                                 excluded=[column]).fit(train_df)
        _, scores = refit_like(ablated, train_df, y_train, val_df, y_val, names, v_weights,
                               v_lambda, v_rho, STEP4, MAX_ITER4, TOL4, decile_edges)
        ablation_change = scores["val_r2"] - v_scores["val_r2"]
        row.update({"ablation_val_r2_change": ablation_change,
                    "ablation_val_rmse_change": scores["val_rmse"] - v_scores["val_rmse"],
                    "ablation_top_decile_shortfall_change":
                        scores["top_decile_shortfall"] - v_scores["top_decile_shortfall"]})
    if column in ADD_BACK_GROUPS:  # add the column (or its main effect) back
        extended = Phase2Features(target_design.phase1,
                                  target_groups + [ADD_BACK_GROUPS[column]]).fit(train_df)
        added = len(extended.feature_names) - len(names)
        all_dropped = added == 0
        row["add_back_columns_kept"] = added
        if not all_dropped:
            _, scores = refit_like(extended, train_df, y_train, val_df, y_val, names, v_weights,
                                   v_lambda, v_rho, STEP4, MAX_ITER4, TOL4, decile_edges)
            add_back_change = scores["val_r2"] - v_scores["val_r2"]
            row.update({"add_back_val_r2_change": add_back_change,
                        "add_back_val_rmse_change": scores["val_rmse"] - v_scores["val_rmse"]})
    A_train, A_val = alone_design(train_df, val_df, column)
    alone = LinearRegressionGD(STEP4 / largest_curvature(A_train), MAX_ITER4, TOL4).fit(A_train, y_train)
    row["alone_val_r2"] = regression_metrics(y_val, A_val @ alone.weights)["r2"]
    row["verdict"] = verdict(in_model, ablation_change, add_back_change, all_dropped,
                             row["alone_val_r2"], VERDICT_R2_TOLERANCE, ALONE_R2_THRESHOLD)
    rows.append(row)
verdicts = pd.DataFrame(rows)
verdicts.to_csv(config.PHASE4_DIR / "column_verdicts.csv", index=False)
display(verdicts.round(4))

In [ ]:
# Artifact for Phase 5: the features the selected Lasso keeps, with lineage.
survivors = [i for i in range(1, len(names)) if lasso_weights[i] != 0]
surviving_names = [names[i] for i in survivors]
surviving_columns = sorted({c for name in surviving_names for c in lineage[name]})
print(f"Lasso keeps {len(survivors)} of {len(names) - 1} features, "
      f"derived from {len(surviving_columns)} original columns: {surviving_columns}")

save_phase4_artifact(config.PHASE4_ARTIFACT, {
    "selected": {label: {"lambda": float(row["lambda"]), "l1_ratio": float(row["l1_ratio"]),
                         **{k: row[k] for k in ("nonzero", "train_r2", "val_r2", "val_rmse",
                                                "top_decile_shortfall",
                                                "top_predicted_decile_shortfall", "status",
                                                "iterations")},
                         "weights": final_weights[label].tolist()}
                 for label, row in final.items()},
    "elastic_net_ratio": float(final["Elastic Net"]["l1_ratio"]),
    "verdict_method": verdict_method,
    "lasso_surviving_features": [{"index": i, "name": names[i], "lineage": lineage[names[i]]}
                                 for i in survivors],
    "lasso_surviving_original_columns": surviving_columns,
    "column_verdicts": verdicts.to_dict(orient="records"),
    "rules": {"select_r2_tolerance": SELECT_R2_TOLERANCE,
              "verdict_r2_tolerance": VERDICT_R2_TOLERANCE,
              "alone_r2_threshold": ALONE_R2_THRESHOLD,
              "grid_sizes": {label: len(grid) for label, (_, grid) in grids.items()},
              "lambda_floor": LAMBDA_FLOOR},
    "settings": {"seed": seed, "step_multiplier": STEP4, "max_iter": MAX_ITER4, "tol": TOL4},
    "features": target_design.to_dict(),
})
print("Saved for Phase 5:", config.PHASE4_ARTIFACT.relative_to(config.ROOT))

### Outcome

_Fill this from the actual pipeline results._

## Phase 5 — Logistic Regression

### Phase 5 plan: label, features, model, cutoff

**Label: what "high demand" means.** An hour is high demand (1) if its `cnt` exceeds the 75th percentile of *training* hours in the same **hour × working-day × year** group (96 groups). The cutoffs come from the training portion only. Validation hours are looked up in that same table, and nothing is recomputed from validation.

Why this definition (numbers in the cell below, training rows only):
- *A single global cutoff mostly teaches the model the clock.* Night hours are never above it and commute peaks nearly always are. The classifier could score well just by reading the hour, and some hours would have one class only (perfect separation, so the logistic weights would grow without bound).
- *A cutoff per hour × working day but not per year mostly labels "is it 2012".* Demand grew between the years, so most 2012 hours would be "high" and most 2011 hours not.
- *Per hour × working day × year*, every kind of hour has about a quarter of its hours labelled high. The label then means "unusually busy for this hour, day type and year": the hours that drain stations the operator would otherwise have stocked for a normal day. Small groups and tied counts move a group's share a little either way.

**Features: exactly Phase 4's Lasso survivors** (selected Lasso after the grid-extension rerun), with the same fitted encoder and scaler and lineage from the Phase 4 artifact. Neither `cnt` nor the group cutoffs ever enter X.

**Model.** $p = \sigma(Xw)$ with $\sigma(z) = 1/(1 + e^{-z})$. We minimize the log-loss $-\frac{1}{m}\sum[y\log p + (1-y)\log(1-p)]$, whose gradient is $X^\top(p - y)/m$, with our own gradient descent loop.
- *Explicit gradient.* $p$ changes at every step, so there is no fixed table of sums as for the squared error.
- *Learning rate from curvature.* The log-loss curvature is $X^\top \mathrm{diag}(p(1-p))X/m$, and $p(1-p) \le 1/4$, so it is at most $L = \lambda_{max}(X^\top X/m)/4$. Gradient descent never increases the loss for any step below $2/L = 8/\lambda_{max}$. We keep the rule used throughout, $1.9/L$.
- *Same stopping rule as before:* max |gradient| < 1e-4, cap 50,000 updates, divergence check. Convergence is reported.

**Probability cutoff.** We pick the cutoff with the best validation F1, searching 0.05–0.95 in steps of 0.01, and show 0.5 for comparison. Choosing the cutoff and scoring it on the same validation set is slightly optimistic: the F1 at the tuned cutoff is a little better than a fresh set would give. ROC-AUC does not depend on the cutoff.

**Report.** Confusion matrix; accuracy next to the always-0 baseline; precision, recall, F1, ROC-AUC; class balance in training and validation. The largest coefficients are read as odds ratios $e^{\beta}$. Each feature is standardized, so $e^{\beta}$ is the factor by which the odds of "high demand" change when that column rises by one standard deviation, all others held fixed.

In [ ]:
from src.phase4_regularization import load_phase4_artifact
from src.phase5_logistic import (
    GROUP_COLUMNS,
    LogisticRegressionGD,
    best_f1_threshold,
    classification_scores,
    high_demand_cutoffs,
    high_demand_labels,
    label_definition_comparison,
    logistic_curvature_bound,
    save_phase5_artifact,
    threshold_scan,
)
from src.visualization import INK, SERIES_COLORS

# Why the label is contextual and includes year (training rows only).
display(label_definition_comparison(train_df).round(3))

# The chosen label: cutoffs from training rows only.
cutoffs = high_demand_cutoffs(train_df)
label_train = high_demand_labels(train_df, cutoffs)
group_positives = (train_df[GROUP_COLUMNS].assign(high=label_train)
                   .groupby(GROUP_COLUMNS)["high"].agg(rows="size", positive_rate="mean")
                   .join(cutoffs))
print(f"training positives: {label_train.sum()} of {len(label_train)} "
      f"({label_train.mean():.1%}) across {len(cutoffs)} groups")
print("group positive rate: min {:.1%}, median {:.1%}, max {:.1%}".format(
    *group_positives["positive_rate"].quantile([0, 0.5, 1])))
display(group_positives.sort_values("positive_rate").round(3))

### Expectation

_Written before the official run._

1. **Separability.** The label removes each hour × working-day × year group's normal level by design, so the clock and calendar cannot predict it on their own. What remains is mainly weather: rain versus sun, temperature, humidity. We expect this to explain most of the "busier than usual" surprises, giving a good ROC-AUC of roughly 0.8–0.9, with accuracy clearly above the 75% always-0 baseline.

2. **Cutoff.** We expect the best-F1 probability cutoff to land above 0.5, favouring precision (fewer false alarms) over recall (fewer missed surges).

3. **Largest odds ratios.** Weather and temperature features should dominate. Hour dummies should matter little, since the label already removes each slot's normal level.

4. **Weakest recall: night hours.** Counts there are tiny and often tied at the cutoff, so a "surge" at 3am is mostly noise that no feature can predict.

In [ ]:
# Features: exactly Phase 4's Lasso survivors, same fitted encoder and scaler.
phase4_artifact = load_phase4_artifact(config.PHASE4_ARTIFACT)
survivors = phase4_artifact["lasso_surviving_features"]
phase5_columns = [0] + [feature["index"] for feature in survivors]  # intercept + survivors
phase5_design = Phase2Features.from_dict(phase4_artifact["features"])
X5_train = phase5_design.transform(train_df)[:, phase5_columns]
X5_val = phase5_design.transform(val_df)[:, phase5_columns]
phase5_names = [phase5_design.feature_names[i] for i in phase5_columns]
assert phase5_names[1:] == [feature["name"] for feature in survivors]
label_val = high_demand_labels(val_df, cutoffs)
print(f"{len(survivors)} Lasso-surviving features (+ intercept); "
      f"class balance: training {label_train.mean():.1%} high, "
      f"validation {label_val.mean():.1%} high")

# Learning rate from the curvature bound: L = lambda_max / 4, step 1.9 / L.
curvature5 = logistic_curvature_bound(X5_train)
step5 = 1.9 / curvature5
print(f"log-loss curvature at most L = {curvature5:.4f}; safe below 2/L = {2 / curvature5:.3f}; "
      f"learning rate 1.9/L = {step5:.3f}")
classifier = LogisticRegressionGD(learning_rate=step5, max_iter=MAX_ITER4, tol=TOL4)
classifier_result = classifier.fit(X5_train, label_train.astype(float))
print(f"{classifier_result.status} after {classifier_result.iterations} updates; "
      f"final max|gradient| {classifier_result.grad_history[-1]:.2e}; loss ever increased: "
      f"{bool(np.any(np.diff(classifier_result.loss_history) > 1e-12))}; "
      f"final training log-loss {classifier_result.loss_history[-1]:.4f}")
plot_gd_runs({"logistic regression": classifier_result}, TOL4,
             "Phase 5: gradient descent on the log-loss (training portion)")
plt.show()

In [ ]:
# Probability cutoff: best validation F1 over 0.05-0.95, and 0.5 for comparison.
prob_val = classifier.predict_proba(X5_val)
scan = threshold_scan(label_val, prob_val, np.round(np.arange(0.05, 0.951, 0.01), 2))
chosen_threshold = best_f1_threshold(scan)
results5 = pd.DataFrame([classification_scores(label_val, prob_val, chosen_threshold),
                         classification_scores(label_val, prob_val, 0.5)],
                        index=[f"tuned cutoff {chosen_threshold:.2f}", "cutoff 0.50"])
print("Validation (the tuned cutoff is chosen and scored on the same set: slightly optimistic)")
display(results5.round(4))
for name, row in results5.iterrows():
    print(f"{name}: confusion matrix (rows = actual 0/1, columns = predicted 0/1)")
    display(pd.DataFrame([[row["tn"], row["fp"]], [row["fn"], row["tp"]]],
                         index=["actual 0", "actual 1"], columns=["predicted 0", "predicted 1"]))

fig, ax = plt.subplots(figsize=(8, 4.5))
for color, column in zip(SERIES_COLORS, ("f1", "precision", "recall")):
    ax.plot(scan["threshold"], scan[column], color=color, lw=2, label=column)
ax.axvline(chosen_threshold, color=INK, ls=":", lw=1.5)
ax.text(chosen_threshold, 0.02, f" tuned {chosen_threshold:.2f}", fontsize=8, color=INK)
ax.set_xlabel("Probability cutoff")
ax.set_ylabel("Validation score")
ax.set_title("Phase 5: choosing the cutoff on validation")
ax.grid(alpha=0.2)
ax.legend()
plt.show()

In [ ]:
# Largest coefficients as odds ratios: e^beta = change in the odds of "high
# demand" when that (standardized) column rises by one standard deviation.
lineage5 = phase5_design.lineage
coefficients5 = pd.DataFrame({
    "feature": phase5_names[1:],
    "from columns": [", ".join(lineage5[name]) for name in phase5_names[1:]],
    "beta": classifier.weights[1:],
})
coefficients5["odds_ratio_per_sd"] = np.exp(coefficients5["beta"])
largest = coefficients5.reindex(coefficients5["beta"].abs().sort_values(ascending=False).index)
print(f"intercept {classifier.weights[0]:.3f}: odds of 'high' for an average hour "
      f"= {np.exp(classifier.weights[0]):.3f}")
display(largest.head(15).round(3))

In [ ]:
# Operator check: which busy hours does the classifier miss? Recall and
# precision at the tuned cutoff by day part and day type (validation).
predicted_val = (prob_val >= chosen_threshold).astype(int)
day_part = pd.cut(val_df["hr"], [-1, 5, 9, 15, 19, 23],
                  labels=["night 0-5", "morning 6-9", "midday 10-15", "evening 16-19", "late 20-23"])
checks = pd.DataFrame({"day part": day_part.to_numpy(), "workingday": val_df["workingday"].to_numpy(),
                       "actual": label_val, "predicted": predicted_val})
def part_scores(group):
    tp = ((group["actual"] == 1) & (group["predicted"] == 1)).sum()
    return pd.Series({"hours": len(group), "high_hours": group["actual"].sum(),
                      "recall": tp / max(group["actual"].sum(), 1),
                      "precision": tp / max(group["predicted"].sum(), 1)})
display(checks.groupby(["workingday", "day part"], observed=True)[["actual", "predicted"]]
        .apply(part_scores).round(3))

In [ ]:
# Diagnostic: how much of the ROC-AUC does the calendar alone explain? (validation)
from sklearn.metrics import roc_auc_score

print("training share of 'high' hours by month (%):",
      (pd.Series(label_train).groupby(train_df["mnth"].to_numpy()).mean() * 100).round(1).to_dict())

# 1. Calendar only, no model: each validation hour is scored by the training
#    share of 'high' hours in its month x weekday.
calendar_rate = (train_df.assign(high=label_train)
                 .groupby(["mnth", "weekday"])["high"].mean().rename("calendar_score"))
calendar_score = val_df.join(calendar_rate, on=["mnth", "weekday"])["calendar_score"].to_numpy()
auc_classifier = roc_auc_score(label_val, prob_val)
auc_calendar = roc_auc_score(label_val, calendar_score)
print(f"ROC-AUC: classifier {auc_classifier:.3f} | calendar only (month x weekday) {auc_calendar:.3f}")

# 2. The classifier inside one month or one season, where the calendar cannot help.
def auc_within(column):
    rows = []
    for value in sorted(val_df[column].unique()):
        mask = (val_df[column] == value).to_numpy()
        both_classes = 0 < label_val[mask].sum() < mask.sum()
        rows.append({column: value, "hours": int(mask.sum()),
                     "positive_rate": label_val[mask].mean(),
                     "roc_auc": roc_auc_score(label_val[mask], prob_val[mask])
                                if both_classes else np.nan})
    return pd.DataFrame(rows)

auc_by_month, auc_by_season = auc_within("mnth"), auc_within("season")
print(f"ROC-AUC within a month, averaged over months: {auc_by_month['roc_auc'].mean():.3f}")
print(f"ROC-AUC within a season: {auc_by_season['roc_auc'].min():.3f} to "
      f"{auc_by_season['roc_auc'].max():.3f}")
display(auc_by_season.round(3))

**What the diagnostic shows.** The label removes each hour × working-day × year group's normal level, but not the season. In the training rows only 0.4% of January hours are "high", against 48.8% of June hours. The label therefore still carries seasonality: part of "unusually busy for this slot and year" simply means "it is summer".

- A ranking that uses only the calendar (month × weekday), with no model, already reaches ROC-AUC **0.809**, against the classifier's **0.871**.
- The classifier's real added skill shows where the calendar cannot help. Ranking hours *within* a single month gives an average ROC-AUC of **0.747**. Within a season it ranges from **0.641** (winter, where only 1.1% of validation hours are positive) to **0.862** (autumn). Temperature, weather and the remaining calendar terms still separate busy from normal hours there, but less well than the headline number suggests.

The headline ROC-AUC should therefore be read together with these two numbers, not alone.

In [ ]:
# Final classifier artifact.
config.PHASE5_DIR.mkdir(parents=True, exist_ok=True)
tuned = results5.iloc[0]
save_phase5_artifact(config.PHASE5_ARTIFACT, {
    "label": {"groups": GROUP_COLUMNS, "quantile": 0.75,
              "cutoffs": [{"hr": int(h), "workingday": int(w), "yr": int(y), "cutoff": float(c)}
                          for (h, w, y), c in cutoffs.items()]},
    "features": phase5_names, "weights": classifier.weights.tolist(),
    "learning_rate": step5, "iterations": classifier_result.iterations,
    "status": classifier_result.status, "threshold": chosen_threshold,
    "validation": results5.to_dict(orient="index"),
})
print("Saved:", config.PHASE5_ARTIFACT.relative_to(config.ROOT))

### Pipeline retrospective

Each phase's key numbers, read from the artifact that phase saved, so the chain is visible in one table: what each phase consumed, what it chose, and what it scored on the seeded validation set.

In [ ]:
# Pipeline retrospective: each phase's key numbers, read from the artifacts it saved.
p1 = load_phase1_artifact(config.PHASE1_ARTIFACT)
p2 = load_phase2_artifact(config.PHASE2_ARTIFACT)
p3 = load_phase3_artifact(config.PHASE3_ARTIFACT)
p4 = load_phase4_artifact(config.PHASE4_ARTIFACT)
retrospective = pd.DataFrame([
    {"phase": "1 Gradient descent", "consumes": "seeded split",
     "key choice": f"47 main effects, learning rate {p1['learning_rate']:.3f}, "
                   f"{p1['iterations']} updates",
     "validation": f"R2 {p1['metrics']['validation']['r2']:.4f}, RMSE {p1['metrics']['validation']['rmse']:.1f}"},
    {"phase": "2 Polynomial", "consumes": "Phase 1 weights (warm start)",
     "key choice": f"degree {p2['degree']}, {len(p2['feature_names']) - 1} features",
     "validation": f"R2 {p2['metrics']['val_r2']:.4f}, RMSE {p2['metrics']['val_rmse']:.1f}"},
    {"phase": "3 Bias-variance", "consumes": "Phase 2 design and order",
     "key choice": f"target up to {p3['target_step']}, {len(p3['feature_names']) - 1} features",
     "validation": f"seeded R2 {p3['diagnosis']['target_val_r2']['seeded']:.4f}, "
                   f"chronological R2 {p3['diagnosis']['target_val_r2']['chronological']:.4f}"},
    *[{"phase": f"4 {label}", "consumes": "Phase 3 target design",
       "key choice": f"lambda {model['lambda']:.3g}, rho {model['l1_ratio']}, "
                     f"{int(model['nonzero'])} nonzero",
       "validation": f"R2 {model['val_r2']:.4f}, RMSE {model['val_rmse']:.1f}"}
      for label, model in p4["selected"].items()],
    {"phase": "5 Logistic", "consumes": f"{len(survivors)} Lasso survivors",
     "key choice": f"contextual label, cutoff {chosen_threshold:.2f}",
     "validation": f"F1 {tuned['f1']:.3f}, ROC-AUC {tuned['roc_auc']:.3f}, "
                   f"accuracy {tuned['accuracy']:.3f} (always-0 {tuned['always_0_accuracy']:.3f})"},
])
display(retrospective)

### Outcome

_Fill this from the actual pipeline results._

## Final model and test predictions

**Model:** Phase 4's recommended regression model, Ridge (λ = 0.00045), on Phase 3's 564-feature design. It has the best seeded validation R² of the three methods. The submission must come from the regression pipeline, not from the Phase 5 classifier.

**Why refit on train + validation.** Every choice was frozen on the seeded split before this step: the features, the interaction groups, the degree, the target design, the penalty and λ, and the GD procedure. The validation rows have done their job of choosing and checking. Refitting on all 10,886 labeled rows changes none of those choices; it only gives the same model 25% more training data. In Phase 3's learning curves, validation error kept falling, slowly, as training rows were added. Everything that is fitted to data is refitted on the combined rows only: encoder levels, scaler, centering means and weights. `test.csv` is used only to predict, never to fit or tune. Predictions below zero are clipped to 0 (rentals cannot be negative; for a nonnegative target, clipping can only reduce squared error). The rows keep the sample file's `instant` order.

In [ ]:
from src.phase4_regularization import PenalizedGD, penalized_learning_rate

# Phase 4's recommended model: Ridge at its selected lambda.
ridge = phase4_artifact["selected"]["Ridge"]
assert ridge["l1_ratio"] == 0.0
final_lambda = ridge["lambda"]

# Refit everything on all labeled rows: encoder, scaler, centering means, weights.
y_all = train_all[config.TARGET].to_numpy(dtype=float)
final_phase1 = Phase1Features(seasonal=phase1_state["seasonal"],
                              use_trend=phase1_state["use_trend"]).fit(train_all)
final_design = Phase2Features(final_phase1, list(phase3_artifact["target_groups"])).fit(train_all)
X_all, X_test = final_design.transform(train_all), final_design.transform(test_X)
print(f"{len(train_all)} labeled rows; {X_all.shape[1] - 1} features "
      f"({len(final_design.dropped)} exact combinations dropped); same feature list as the "
      f"seeded-split design: {final_design.feature_names == target_design.feature_names}")

# Same GD procedure as Phase 4: the Ridge path from its largest lambda down to the
# selected one, each fit starting from the previous fit's weights.
ridge_grid = grids["Ridge"][1]
stop = int(np.flatnonzero(np.isclose(ridge_grid, final_lambda, rtol=1e-12, atol=0))[0])
curvature_all = largest_curvature(X_all)
final_weights_all = np.zeros(X_all.shape[1])
final_weights_all[0] = y_all.mean()
for lam in ridge_grid[: stop + 1]:
    final_fit = PenalizedGD(lam, 0.0, penalized_learning_rate(curvature_all, lam, 0.0, STEP4),
                            MAX_ITER4, TOL4).fit(X_all, y_all, init_weights=final_weights_all)
    assert final_fit.converged, f"Ridge refit did not converge at lambda {lam}"
    final_weights_all = final_fit.weights
fit_scores = regression_metrics(y_all, X_all @ final_weights_all)
print(f"final fit (lambda {final_lambda:.3g}): {final_fit.status} after {final_fit.iterations} "
      f"updates, final max|gradient| {final_fit.grad_history[-1]:.2e}; "
      f"training R2 {fit_scores['r2']:.4f}, RMSE {fit_scores['rmse']:.2f} on all labeled rows")

# Predict test.csv, clip below zero, write in the sample file's format and order.
raw_test = X_test @ final_weights_all
sample = pd.read_csv(config.SAMPLE_SUBMISSION_PATH)
assert list(sample.columns) == ["instant", "cnt"]
assert list(sample["instant"]) == list(test_X["instant"]), "test rows must follow the sample order"
submission = sample[["instant"]].assign(cnt=np.clip(raw_test, 0, None))
assert len(submission) == 574 and np.isfinite(submission["cnt"]).all()
assert (submission["cnt"] >= 0).all() and submission["instant"].is_unique
config.SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
submission.to_csv(config.SUBMISSION_PATH, index=False)
print(f"{len(submission)} predictions; {int(np.sum(raw_test < 0))} below zero clipped to 0; "
      f"min {submission['cnt'].min():.1f}, mean {submission['cnt'].mean():.1f}, "
      f"max {submission['cnt'].max():.1f}")
print("Saved:", config.SUBMISSION_PATH.relative_to(config.ROOT))